In [ ]:
!pip install dnspython pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 7.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import os

BASE_DIR = "/content/drive/MyDrive/AI_Email_Deliverability_Intelligence"

external_dir = os.path.join(
    BASE_DIR,
    "data",
    "external"
)

os.makedirs(external_dir, exist_ok=True)

test_domains = pd.DataFrame({
    "domain": [
        "google.com",
        "microsoft.com",
        "amazon.com",
        "github.com",
        "openai.com"
    ]
})

domains_path = os.path.join(
    external_dir,
    "domains_to_scan.csv"
)

test_domains.to_csv(
    domains_path,
    index=False
)

print("Domain input file created:")
print(domains_path)

display(test_domains)

Domain input file created:
/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/external/domains_to_scan.csv


,domain
0,google.com
1,microsoft.com
2,amazon.com
3,github.com
4,openai.com


In [ ]:
import dns.resolver
import pandas as pd
from datetime import datetime, timezone

domains = pd.read_csv(domains_path)

resolver = dns.resolver.Resolver()

results = []

for domain in domains["domain"].dropna().unique():

    record = {
        "domain": domain,
        "checked_at_utc": datetime.now(timezone.utc).isoformat(),
        "mx_records": None,
        "spf_record": None,
        "dmarc_record": None,
        "a_records": None,
        "aaaa_records": None
    }

    # -----------------------------
    # MX
    # -----------------------------
    try:
        answers = resolver.resolve(domain, "MX")

        record["mx_records"] = "; ".join(
            sorted(
                str(answer.exchange).rstrip(".")
                for answer in answers
            )
        )
    except Exception:
        pass

    # -----------------------------
    # TXT → SPF
    # -----------------------------
    try:
        answers = resolver.resolve(domain, "TXT")

        txt_records = [
            "".join(
                part.decode("utf-8")
                if isinstance(part, bytes)
                else part
                for part in answer.strings
            )
            for answer in answers
        ]

        spf_records = [
            txt for txt in txt_records
            if txt.lower().startswith("v=spf1")
        ]

        if spf_records:
            record["spf_record"] = " | ".join(spf_records)

    except Exception:
        pass

    # -----------------------------
    # DMARC
    # -----------------------------
    try:
        answers = resolver.resolve(
            f"_dmarc.{domain}",
            "TXT"
        )

        txt_records = [
            "".join(
                part.decode("utf-8")
                if isinstance(part, bytes)
                else part
                for part in answer.strings
            )
            for answer in answers
        ]

        dmarc_records = [
            txt for txt in txt_records
            if txt.lower().startswith("v=dmarc1")
        ]

        if dmarc_records:
            record["dmarc_record"] = " | ".join(
                dmarc_records
            )

    except Exception:
        pass

    # -----------------------------
    # A
    # -----------------------------
    try:
        answers = resolver.resolve(domain, "A")

        record["a_records"] = "; ".join(
            sorted(str(answer) for answer in answers)
        )
    except Exception:
        pass

    # -----------------------------
    # AAAA
    # -----------------------------
    try:
        answers = resolver.resolve(domain, "AAAA")

        record["aaaa_records"] = "; ".join(
            sorted(str(answer) for answer in answers)
        )
    except Exception:
        pass

    results.append(record)

dns_results = pd.DataFrame(results)

display(dns_results)

,domain,checked_at_utc,mx_records,spf_record,dmarc_record,a_records,aaaa_records
0,google.com,2026-09-22T08:12:38.711476+00:00,smtp.google.com,None,v=DMARC1; p=reject; rua=mailto:mailauth-report...,172.217.214.100; 172.217.214.101; 172.217.214....,2607:f8b0:4001:c05::64; 2607:f8b0:4001:c05::71...
1,microsoft.com,2026-09-22T08:12:44.162350+00:00,microsoft-com.mail.protection.outlook.com,None,v=DMARC1; p=reject; pct=100; rua=mailto:itex-r...,150.171.110.211,2603:1061:14:1d2::1
2,amazon.com,2026-09-22T08:12:49.612987+00:00,amazon-smtp.amazon.com,None,v=DMARC1;p=quarantine;pct=100;rua=mailto:repor...,98.82.161.185; 98.87.170.71; 98.87.170.74,None
3,github.com,2026-09-22T08:12:55.100140+00:00,github-com.mail.protection.outlook.com,None,v=DMARC1; p=quarantine; sp=reject; pct=100; ru...,140.82.112.3,None
4,openai.com,2026-09-22T08:13:00.580361+00:00,alt1.aspmx.l.google.com; alt2.aspmx.l.google.c...,None,v=DMARC1; p=reject; rua=mailto:tdfyvl0n@ag.dma...,104.18.33.45; 172.64.154.211,None


In [ ]:
dns_results_path = os.path.join(
    external_dir,
    "dns_observations.csv"
)

dns_results.to_csv(
    dns_results_path,
    index=False
)

print("DNS observations saved:")
print(dns_results_path)

DNS observations saved:
/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/external/dns_observations.csv


In [ ]:
display(dns_results)

,domain,checked_at_utc,mx_records,spf_record,dmarc_record,a_records,aaaa_records
0,google.com,2026-09-22T08:12:38.711476+00:00,smtp.google.com,None,v=DMARC1; p=reject; rua=mailto:mailauth-report...,172.217.214.100; 172.217.214.101; 172.217.214....,2607:f8b0:4001:c05::64; 2607:f8b0:4001:c05::71...
1,microsoft.com,2026-09-22T08:12:44.162350+00:00,microsoft-com.mail.protection.outlook.com,None,v=DMARC1; p=reject; pct=100; rua=mailto:itex-r...,150.171.110.211,2603:1061:14:1d2::1
2,amazon.com,2026-09-22T08:12:49.612987+00:00,amazon-smtp.amazon.com,None,v=DMARC1;p=quarantine;pct=100;rua=mailto:repor...,98.82.161.185; 98.87.170.71; 98.87.170.74,None
3,github.com,2026-09-22T08:12:55.100140+00:00,github-com.mail.protection.outlook.com,None,v=DMARC1; p=quarantine; sp=reject; pct=100; ru...,140.82.112.3,None
4,openai.com,2026-09-22T08:13:00.580361+00:00,alt1.aspmx.l.google.com; alt2.aspmx.l.google.c...,None,v=DMARC1; p=reject; rua=mailto:tdfyvl0n@ag.dma...,104.18.33.45; 172.64.154.211,None


In [ ]:
print(dns_results.columns.tolist())

['domain', 'checked_at_utc', 'mx_records', 'spf_record', 'dmarc_record', 'a_records', 'aaaa_records']


In [ ]:
# ============================================
# STRUCTURED DNS / AUTHENTICATION FEATURES
# ============================================

import re
import pandas as pd
import numpy as np

structured_dns = dns_results.copy()

# --------------------------------------------
# SPF
# --------------------------------------------

structured_dns["spf_present"] = (
    structured_dns["spf_record"].notna()
)

structured_dns["spf_record_count"] = (
    structured_dns["spf_record"]
    .fillna("")
    .apply(
        lambda x: len(
            [r for r in x.split(" | ") if r.strip()]
        )
    )
)

# --------------------------------------------
# DMARC
# --------------------------------------------

structured_dns["dmarc_present"] = (
    structured_dns["dmarc_record"].notna()
)

def extract_dmarc_tag(record, tag):
    if pd.isna(record):
        return None

    match = re.search(
        rf"(?:^|;){tag}=([^;]+)",
        str(record),
        flags=re.IGNORECASE
    )

    return match.group(1).strip() if match else None


structured_dns["dmarc_policy"] = (
    structured_dns["dmarc_record"]
    .apply(lambda x: extract_dmarc_tag(x, "p"))
)

structured_dns["dmarc_subdomain_policy"] = (
    structured_dns["dmarc_record"]
    .apply(lambda x: extract_dmarc_tag(x, "sp"))
)

structured_dns["dmarc_percentage"] = (
    pd.to_numeric(
        structured_dns["dmarc_record"]
        .apply(lambda x: extract_dmarc_tag(x, "pct")),
        errors="coerce"
    )
)

# --------------------------------------------
# MX
# --------------------------------------------

structured_dns["mx_present"] = (
    structured_dns["mx_records"].notna()
)

structured_dns["mx_record_count"] = (
    structured_dns["mx_records"]
    .fillna("")
    .apply(
        lambda x: len(
            [r for r in x.split(";") if r.strip()]
        )
    )
)

# --------------------------------------------
# A / AAAA
# --------------------------------------------

structured_dns["a_record_count"] = (
    structured_dns["a_records"]
    .fillna("")
    .apply(
        lambda x: len(
            [r for r in x.split(";") if r.strip()]
        )
    )
)

structured_dns["aaaa_record_count"] = (
    structured_dns["aaaa_records"]
    .fillna("")
    .apply(
        lambda x: len(
            [r for r in x.split(";") if r.strip()]
        )
    )
)

# --------------------------------------------
# Authentication summary
# --------------------------------------------

structured_dns["authentication_records_available"] = (
    structured_dns[
        [
            "spf_present",
            "dmarc_present"
        ]
    ]
    .sum(axis=1)
)

display(structured_dns)

,domain,checked_at_utc,mx_records,spf_record,dmarc_record,a_records,aaaa_records,spf_present,spf_record_count,dmarc_present,dmarc_policy,dmarc_subdomain_policy,dmarc_percentage,mx_present,mx_record_count,a_record_count,aaaa_record_count,authentication_records_available
0,google.com,2026-09-22T08:12:38.711476+00:00,smtp.google.com,None,v=DMARC1; p=reject; rua=mailto:mailauth-report...,172.217.214.100; 172.217.214.101; 172.217.214....,2607:f8b0:4001:c05::64; 2607:f8b0:4001:c05::71...,False,0,True,None,None,NaN,True,1,6,4,1
1,microsoft.com,2026-09-22T08:12:44.162350+00:00,microsoft-com.mail.protection.outlook.com,None,v=DMARC1; p=reject; pct=100; rua=mailto:itex-r...,150.171.110.211,2603:1061:14:1d2::1,False,0,True,None,None,NaN,True,1,1,1,1
2,amazon.com,2026-09-22T08:12:49.612987+00:00,amazon-smtp.amazon.com,None,v=DMARC1;p=quarantine;pct=100;rua=mailto:repor...,98.82.161.185; 98.87.170.71; 98.87.170.74,None,False,0,True,quarantine,None,100.0,True,1,3,0,1
3,github.com,2026-09-22T08:12:55.100140+00:00,github-com.mail.protection.outlook.com,None,v=DMARC1; p=quarantine; sp=reject; pct=100; ru...,140.82.112.3,None,False,0,True,None,None,NaN,True,1,1,0,1
4,openai.com,2026-09-22T08:13:00.580361+00:00,alt1.aspmx.l.google.com; alt2.aspmx.l.google.c...,None,v=DMARC1; p=reject; rua=mailto:tdfyvl0n@ag.dma...,104.18.33.45; 172.64.154.211,None,False,0,True,None,None,NaN,True,5,2,0,1


In [ ]:
structured_dns_path = os.path.join(
    external_dir,
    "dns_authentication_observations.csv"
)

structured_dns.to_csv(
    structured_dns_path,
    index=False
)

print("Structured DNS data saved:")
print(structured_dns_path)

Structured DNS data saved:
/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/external/dns_authentication_observations.csv


In [ ]:
import requests
import zipfile
import io
import pandas as pd
import os
from datetime import datetime, timezone

BASE_DIR = "/content/drive/MyDrive/AI_Email_Deliverability_Intelligence"

external_dir = os.path.join(
    BASE_DIR,
    "data",
    "external"
)

os.makedirs(external_dir, exist_ok=True)

tranco_url = "https://tranco-list.eu/top-1m.csv.zip"

response = requests.get(tranco_url)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    print("Files in archive:")
    print(z.namelist())

    csv_name = z.namelist()[0]

    with z.open(csv_name) as f:
        tranco = pd.read_csv(
            f,
            header=None,
            names=["rank", "domain"]
        )

print("Downloaded domains:", len(tranco))

tranco.head()

Files in archive:
['top-1m.csv']
Downloaded domains: 1000000


,rank,domain
0,1,google.com
1,2,cloudflare.com
2,3,facebook.com
3,4,gstatic.com
4,5,googleapis.com


In [ ]:
domain_population = (
    tranco
    .head(10000)
    .copy()
)

domain_population["population_source"] = "Tranco"
domain_population["population_rank"] = domain_population["rank"]
domain_population["collected_at_utc"] = (
    datetime.now(timezone.utc).isoformat()
)

domain_population = domain_population[
    [
        "domain",
        "population_source",
        "population_rank",
        "collected_at_utc"
    ]
]

display(domain_population.head(20))
print("Population size:", len(domain_population))

,domain,population_source,population_rank,collected_at_utc
0,google.com,Tranco,1,2026-09-22T08:13:07.837138+00:00
1,cloudflare.com,Tranco,2,2026-09-22T08:13:07.837138+00:00
2,facebook.com,Tranco,3,2026-09-22T08:13:07.837138+00:00
3,gstatic.com,Tranco,4,2026-09-22T08:13:07.837138+00:00
4,googleapis.com,Tranco,5,2026-09-22T08:13:07.837138+00:00
5,microsoft.com,Tranco,6,2026-09-22T08:13:07.837138+00:00
6,akamai.net,Tranco,7,2026-09-22T08:13:07.837138+00:00
7,amazonaws.com,Tranco,8,2026-09-22T08:13:07.837138+00:00
8,youtube.com,Tranco,9,2026-09-22T08:13:07.837138+00:00
9,apple.com,Tranco,10,2026-09-22T08:13:07.837138+00:00


Population size: 10000


In [ ]:
domain_population_path = os.path.join(
    external_dir,
    "domain_population_10000.csv"
)

domain_population.to_csv(
    domain_population_path,
    index=False
)

print("Saved:")
print(domain_population_path)

Saved:
/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/external/domain_population_10000.csv


In [ ]:
# ============================================
# FIXED TXT / SPF / DMARC PARSER
# ============================================

import pandas as pd
import dns.resolver
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
import os
import time

domains_df = pd.read_csv(domain_population_path)

domains = (
    domains_df["domain"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
    .drop_duplicates()
    .tolist()
)

print("Domains to scan:", len(domains))


def clean_txt_rdata(rdata):
    """
    Convert dnspython TXT RDATA into a clean string.
    TXT records may be represented as one or more quoted chunks.
    """
    text = rdata.to_text()

    # Remove quotes between adjacent TXT chunks
    text = text.replace('" "', '')

    # Remove surrounding quotes
    text = text.strip('"')

    return text.strip()


def lookup_dns(resolver, name, record_type):

    try:
        answers = resolver.resolve(
            name,
            record_type
        )

        values = []

        for answer in answers:

            if record_type == "MX":
                values.append(
                    str(answer.exchange).rstrip(".")
                )

            elif record_type == "TXT":
                values.append(
                    clean_txt_rdata(answer)
                )

            else:
                values.append(str(answer))

        return values, None

    except (
        dns.resolver.NoAnswer,
        dns.resolver.NXDOMAIN,
        dns.resolver.NoNameservers,
        dns.exception.Timeout
    ) as e:

        return [], type(e).__name__

    except Exception as e:

        return [], type(e).__name__


def scan_domain_fixed(domain):

    resolver = dns.resolver.Resolver()

    resolver.timeout = 3
    resolver.lifetime = 5

    checked_at = datetime.now(
        timezone.utc
    ).isoformat()

    result = {
        "domain": domain,
        "checked_at_utc": checked_at,

        "mx_present": False,
        "mx_records": None,
        "mx_error": None,

        "spf_present": False,
        "spf_record": None,
        "spf_error": None,

        "dmarc_present": False,
        "dmarc_record": None,
        "dmarc_error": None,

        "a_records": None,
        "a_error": None,

        "aaaa_records": None,
        "aaaa_error": None
    }

    # ----------------------------------------
    # MX
    # ----------------------------------------

    mx_values, mx_error = lookup_dns(
        resolver,
        domain,
        "MX"
    )

    if mx_values:
        result["mx_present"] = True
        result["mx_records"] = "; ".join(
            sorted(set(mx_values))
        )

    result["mx_error"] = mx_error

    # ----------------------------------------
    # TXT → SPF
    # ----------------------------------------

    txt_values, txt_error = lookup_dns(
        resolver,
        domain,
        "TXT"
    )

    spf_values = [
        txt
        for txt in txt_values
        if txt.lower().startswith("v=spf1")
    ]

    if spf_values:
        result["spf_present"] = True
        result["spf_record"] = " | ".join(
            sorted(set(spf_values))
        )

    result["spf_error"] = txt_error

    # ----------------------------------------
    # DMARC
    # ----------------------------------------

    dmarc_values, dmarc_error = lookup_dns(
        resolver,
        f"_dmarc.{domain}",
        "TXT"
    )

    dmarc_values = [
        txt
        for txt in dmarc_values
        if txt.lower().startswith("v=dmarc1")
    ]

    if dmarc_values:
        result["dmarc_present"] = True
        result["dmarc_record"] = " | ".join(
            sorted(set(dmarc_values))
        )

    result["dmarc_error"] = dmarc_error

    # ----------------------------------------
    # A
    # ----------------------------------------

    a_values, a_error = lookup_dns(
        resolver,
        domain,
        "A"
    )

    if a_values:
        result["a_records"] = "; ".join(
            sorted(set(a_values))
        )

    result["a_error"] = a_error

    # ----------------------------------------
    # AAAA
    # ----------------------------------------

    aaaa_values, aaaa_error = lookup_dns(
        resolver,
        domain,
        "AAAA"
    )

    if aaaa_values:
        result["aaaa_records"] = "; ".join(
            sorted(set(aaaa_values))
        )

    result["aaaa_error"] = aaaa_error

    return result


# ============================================
# Scan 10,000 domains
# ============================================

results_fixed = []

start_time = time.time()

MAX_WORKERS = 10

with ThreadPoolExecutor(
    max_workers=MAX_WORKERS
) as executor:

    futures = {
        executor.submit(
            scan_domain_fixed,
            domain
        ): domain
        for domain in domains
    }

    completed = 0

    for future in as_completed(futures):

        domain = futures[future]

        try:
            results_fixed.append(
                future.result()
            )

        except Exception as e:

            results_fixed.append({
                "domain": domain,
                "checked_at_utc": datetime.now(
                    timezone.utc
                ).isoformat(),

                "mx_present": False,
                "mx_records": None,
                "mx_error": str(e),

                "spf_present": False,
                "spf_record": None,
                "spf_error": str(e),

                "dmarc_present": False,
                "dmarc_record": None,
                "dmarc_error": str(e),

                "a_records": None,
                "a_error": str(e),

                "aaaa_records": None,
                "aaaa_error": str(e)
            })

        completed += 1

        if completed % 500 == 0:

            elapsed = time.time() - start_time

            print(
                f"Completed: {completed}/{len(domains)} "
                f"| {elapsed:.1f}s"
            )


dns_10k_fixed = (
    pd.DataFrame(results_fixed)
    .sort_values("domain")
    .reset_index(drop=True)
)

print("\nScan completed.")
print("Rows:", len(dns_10k_fixed))

Domains to scan: 10000
Completed: 500/10000 | 121.3s
Completed: 1000/10000 | 265.4s
Completed: 1500/10000 | 398.0s
Completed: 2000/10000 | 488.9s
Completed: 2500/10000 | 622.7s
Completed: 3000/10000 | 719.2s
Completed: 3500/10000 | 841.8s
Completed: 4000/10000 | 960.8s
Completed: 4500/10000 | 1103.6s
Completed: 5000/10000 | 1226.5s
Completed: 5500/10000 | 1352.0s
Completed: 6000/10000 | 1483.7s
Completed: 6500/10000 | 1626.1s
Completed: 7000/10000 | 1752.5s
Completed: 7500/10000 | 1876.1s
Completed: 8000/10000 | 2000.6s
Completed: 8500/10000 | 2103.3s
Completed: 9000/10000 | 2194.1s
Completed: 9500/10000 | 2272.0s
Completed: 10000/10000 | 2341.8s

Scan completed.
Rows: 10000


In [ ]:
print("Total domains:", len(dns_10k_fixed))

print(
    "MX present:",
    dns_10k_fixed["mx_present"].sum()
)

print(
    "SPF present:",
    dns_10k_fixed["spf_present"].sum()
)

print(
    "DMARC present:",
    dns_10k_fixed["dmarc_present"].sum()
)

print(
    "A present:",
    dns_10k_fixed["a_records"].notna().sum()
)

print(
    "AAAA present:",
    dns_10k_fixed["aaaa_records"].notna().sum()
)

Total domains: 10000
MX present: 7283
SPF present: 3749
DMARC present: 6705
A present: 8345
AAAA present: 3025


In [ ]:
print("\nSample SPF records:")
display(
    dns_10k_fixed.loc[
        dns_10k_fixed["spf_present"],
        ["domain", "spf_record"]
    ].head(10)
)

print("\nSample DMARC records:")
display(
    dns_10k_fixed.loc[
        dns_10k_fixed["dmarc_present"],
        ["domain", "dmarc_record"]
    ].head(10)
)


Sample SPF records:


,domain,spf_record
0,0123tt.ru,v=spf1 ip4:188.120.230.141 a mx ~all
5,10jqka.com.cn,v=spf1 ip4:220.189.211.0/27 ip4:122.224.106.0/...
18,163.com,v=spf1 include:spf.mail.163.com -all
22,1688.com,v=spf1 include:spf1.staff.mail.aliyun.com -all
23,17track.net,v=spf1 include:spf.mail.qq.com ~all
26,189.cn,v=spf1 include:mail-spf.21cn.com include:hwmai...
28,1984.is,v=spf1 include:_spf.myorderbox.com ip4:93.95.2...
29,1c.ru,v=spf1 +mx a:mailrelay.it-lite.ru ip4:185.12.1...
30,1capp.com,v=spf1 mx mx:mx.1cbo.ru ip4:95.163.180.195/32 ...
31,1cbit.ru,v=spf1 include:_spf.1cbit.ru include:_spf.dash...



Sample DMARC records:


,domain,dmarc_record
9,123rf.com,v=DMARC1; p=quarantine; rua=mailto:postmaster@...
17,15min.lt,v=DMARC1; p=quarantine; pct=100;
18,163.com,v=DMARC1; p=none;
23,17track.net,v=DMARC1;p=quarantine;adkim=r;aspf=r;rua=mailt...
28,1984.is,v=DMARC1; p=none
29,1c.ru,v=DMARC1; p=reject; pct=100; sp=none; rua=mail...
30,1capp.com,v=DMARC1; p=none; rua=mailto:admin@1capp.com
31,1cbit.ru,v=DMARC1; p=none; ruf=mailto:dmark_info@1cbit....
33,1digitech.com,v=DMARC1; p=reject; sp=reject; adkim=s; aspf=s...
34,1drv.com,v=DMARC1; p=reject; sp=reject; pct=100; rua=ma...


In [ ]:
dns_10k_path = os.path.join(
    external_dir,
    "dns_observations_10000.csv"
)

dns_10k_fixed.to_csv(
    dns_10k_path,
    index=False
)

print("Corrected DNS observations saved:")
print(dns_10k_path)

Corrected DNS observations saved:
/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/external/dns_observations_10000.csv


In [ ]:
# ============================================
# ROBUST SPF + DMARC PARSER
# ============================================

import re
import pandas as pd
import numpy as np

dns_features = dns_10k_fixed.copy()


# ============================================
# SPF
# ============================================

def split_records(value):
    if pd.isna(value) or not str(value).strip():
        return []

    return [
        r.strip().strip('"')
        for r in str(value).split(" | ")
        if r.strip()
    ]


def parse_spf(record_string):

    records = split_records(record_string)

    spf_records = [
        r for r in records
        if r.lower().startswith("v=spf1")
    ]

    result = {
        "spf_record_count": len(spf_records),
        "spf_multiple_records": len(spf_records) > 1,
        "spf_has_all": False,
        "spf_all_qualifier": None,
        "spf_include_count": 0,
        "spf_ip4_count": 0,
        "spf_ip6_count": 0,
        "spf_a_present": False,
        "spf_mx_present": False,
        "spf_redirect_present": False
    }

    # Multiple SPF records are not a normal valid
    # single-policy situation, so keep them flagged.
    if len(spf_records) != 1:
        return result

    record = spf_records[0]

    tokens = record.split()

    # ----------------------------------------
    # Count mechanisms
    # ----------------------------------------

    result["spf_include_count"] = sum(
        token.lower().lstrip("+-~?")
        .startswith("include:")
        for token in tokens
    )

    result["spf_ip4_count"] = sum(
        token.lower().lstrip("+-~?")
        .startswith("ip4:")
        for token in tokens
    )

    result["spf_ip6_count"] = sum(
        token.lower().lstrip("+-~?")
        .startswith("ip6:")
        for token in tokens
    )

    result["spf_a_present"] = any(
        token.lower().lstrip("+-~?") == "a"
        or token.lower().lstrip("+-~?").startswith("a:")
        for token in tokens
    )

    result["spf_mx_present"] = any(
        token.lower().lstrip("+-~?") == "mx"
        or token.lower().lstrip("+-~?").startswith("mx:")
        for token in tokens
    )

    result["spf_redirect_present"] = any(
        token.lower().startswith("redirect=")
        for token in tokens
    )

    # ----------------------------------------
    # Find explicit ALL mechanism
    # ----------------------------------------

    for token in tokens:

        match = re.fullmatch(
            r"([+\-~?]?)all",
            token,
            flags=re.IGNORECASE
        )

        if match:
            qualifier = match.group(1)

            result["spf_has_all"] = True

            # '+' is the explicit pass qualifier;
            # an omitted qualifier has the same meaning.
            result["spf_all_qualifier"] = (
                qualifier if qualifier else "+"
            )

            break

    return result


spf_parsed = (
    dns_features["spf_record"]
    .apply(parse_spf)
    .apply(pd.Series)
)

dns_features = pd.concat(
    [
        dns_features,
        spf_parsed
    ],
    axis=1
)

dns_features["spf_present"] = (
    dns_features["spf_record_count"] > 0
)


# ============================================
# DMARC
# ============================================

def parse_dmarc(record_string):

    records = split_records(record_string)

    dmarc_records = [
        r for r in records
        if r.lower().startswith("v=dmarc1")
    ]

    result = {
        "dmarc_record_count": len(dmarc_records),
        "dmarc_multiple_records": len(dmarc_records) > 1,
        "dmarc_policy": None,
        "dmarc_subdomain_policy": None,
        "dmarc_percentage": None,
        "dmarc_adkim": None,
        "dmarc_aspf": None,
        "dmarc_fo": None,
        "dmarc_reporting_enabled": False
    }

    # Only parse a single clear DMARC record.
    if len(dmarc_records) != 1:
        return result

    record = dmarc_records[0]

    # ----------------------------------------
    # Split into semicolon-separated tags
    # ----------------------------------------

    tags = {}

    for part in record.split(";"):

        part = part.strip()

        if "=" not in part:
            continue

        key, value = part.split(
            "=",
            1
        )

        tags[key.strip().lower()] = (
            value.strip()
        )

    # ----------------------------------------
    # Policy
    # ----------------------------------------

    policy = tags.get("p")

    if policy in {
        "none",
        "quarantine",
        "reject"
    }:
        result["dmarc_policy"] = policy

    # ----------------------------------------
    # Subdomain policy
    # ----------------------------------------

    sub_policy = tags.get("sp")

    if sub_policy in {
        "none",
        "quarantine",
        "reject"
    }:
        result["dmarc_subdomain_policy"] = (
            sub_policy
        )

    # ----------------------------------------
    # Percentage
    # ----------------------------------------

    if "pct" in tags:

        try:
            pct = float(tags["pct"])

            if 0 <= pct <= 100:
                result["dmarc_percentage"] = pct

        except ValueError:
            pass

    # ----------------------------------------
    # Alignment
    # ----------------------------------------

    if tags.get("adkim") in {"r", "s"}:
        result["dmarc_adkim"] = tags["adkim"]

    if tags.get("aspf") in {"r", "s"}:
        result["dmarc_aspf"] = tags["aspf"]

    if "fo" in tags:
        result["dmarc_fo"] = tags["fo"]

    # ----------------------------------------
    # Aggregate reporting
    # ----------------------------------------

    result["dmarc_reporting_enabled"] = (
        "rua" in tags
        and bool(tags["rua"])
    )

    return result


dmarc_parsed = (
    dns_features["dmarc_record"]
    .apply(parse_dmarc)
    .apply(pd.Series)
)

dns_features = pd.concat(
    [
        dns_features,
        dmarc_parsed
    ],
    axis=1
)

dns_features["dmarc_present"] = (
    dns_features["dmarc_record_count"] > 0
)


# ============================================
# MX / IP features
# ============================================

dns_features["mx_record_count"] = (
    dns_features["mx_records"]
    .fillna("")
    .apply(
        lambda x: len([
            r for r in x.split(";")
            if r.strip()
        ])
    )
)

dns_features["a_record_count"] = (
    dns_features["a_records"]
    .fillna("")
    .apply(
        lambda x: len([
            r for r in x.split(";")
            if r.strip()
        ])
    )
)

dns_features["aaaa_record_count"] = (
    dns_features["aaaa_records"]
    .fillna("")
    .apply(
        lambda x: len([
            r for r in x.split(";")
            if r.strip()
        ])
    )
)

display(dns_features.head())

,domain,checked_at_utc,mx_present,mx_records,mx_error,spf_present,spf_record,spf_error,dmarc_present,dmarc_record,...,dmarc_policy,dmarc_subdomain_policy,dmarc_percentage,dmarc_adkim,dmarc_aspf,dmarc_fo,dmarc_reporting_enabled,mx_record_count,a_record_count,aaaa_record_count
0,0123tt.ru,2026-09-22T08:44:56.961348+00:00,True,mail.0123tt.ru,None,True,v=spf1 ip4:188.120.230.141 a mx ~all,None,False,None,...,None,None,NaN,None,None,None,False,1,1,0
1,0lin.com,2026-09-22T08:51:20.915164+00:00,False,None,NoAnswer,False,None,NoAnswer,False,None,...,None,None,NaN,None,None,None,False,0,2,2
2,0xrpc.io,2026-09-22T08:25:52.016166+00:00,False,None,NoAnswer,False,None,NoAnswer,False,None,...,None,None,NaN,None,None,None,False,0,1,0
3,10086.cn,2026-09-22T08:29:28.803440+00:00,True,mx.139.com,None,False,None,NoAnswer,False,None,...,None,None,NaN,None,None,None,False,1,2,2
4,1024tera.com,2026-09-22T08:47:15.422021+00:00,False,None,NoAnswer,False,None,None,False,None,...,None,None,NaN,None,None,None,False,0,1,0


In [ ]:
structured_dns_path = os.path.join(
    external_dir,
    "dns_authentication_features_10000.csv"
)

dns_features.to_csv(
    structured_dns_path,
    index=False
)

print("Saved:")
print(structured_dns_path)

Saved:
/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/external/dns_authentication_features_10000.csv


In [ ]:
print("\nDMARC policy distribution:")
print(
    dns_features["dmarc_policy"]
    .value_counts(dropna=False)
)

print("\nInvalid DMARC policies:")
print(
    dns_features[
        dns_features["dmarc_policy"].notna()
        &
        ~dns_features["dmarc_policy"].isin(
            ["none", "quarantine", "reject"]
        )
    ][
        ["domain", "dmarc_record", "dmarc_policy"]
    ]
)

print("\nSPF ALL qualifier:")
print(
    dns_features["spf_all_qualifier"]
    .value_counts(dropna=False)
)

print("\nMultiple SPF records:")
print(
    dns_features["spf_multiple_records"].sum()
)

print("\nMultiple DMARC records:")
print(
    dns_features["dmarc_multiple_records"].sum()
)


DMARC policy distribution:
dmarc_policy
None          3322
reject        3149
quarantine    1862
none          1667
Name: count, dtype: int64

Invalid DMARC policies:
Empty DataFrame
Columns: [domain, dmarc_record, dmarc_policy]
Index: []

SPF ALL qualifier:
spf_all_qualifier
None    6425
-       1930
~       1545
?         97
+          3
Name: count, dtype: int64

Multiple SPF records:
33

Multiple DMARC records:
19


In [ ]:
structured_dns_path = os.path.join(
    external_dir,
    "dns_authentication_features_10000.csv"
)

dns_features.to_csv(
    structured_dns_path,
    index=False
)

print("Validated structured DNS dataset saved:")
print(structured_dns_path)

Validated structured DNS dataset saved:
/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/external/dns_authentication_features_10000.csv


In [ ]:
print("Rows:", len(dns_features))
print("Columns:", len(dns_features.columns))

print(
    "Invalid DMARC policies:",
    (
        dns_features["dmarc_policy"].notna()
        &
        ~dns_features["dmarc_policy"].isin(
            ["none", "quarantine", "reject"]
        )
    ).sum()
)

print(
    "Multiple SPF records:",
    dns_features["spf_multiple_records"].sum()
)

print(
    "Multiple DMARC records:",
    dns_features["dmarc_multiple_records"].sum()
)

Rows: 10000
Columns: 37
Invalid DMARC policies: 0
Multiple SPF records: 33
Multiple DMARC records: 19


In [ ]:
structured_dns_path = os.path.join(
    external_dir,
    "dns_authentication_features_10000.csv"
)

dns_features.to_csv(
    structured_dns_path,
    index=False
)

print("Saved successfully:")
print(structured_dns_path)

Saved successfully:
/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/external/dns_authentication_features_10000.csv


In [ ]:
structured_dns_path = os.path.join(
    external_dir,
    "dns_authentication_features_10000.csv"
)

dns_features.to_csv(
    structured_dns_path,
    index=False
)

print("Saved successfully:")
print(structured_dns_path)

Saved successfully:
/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/external/dns_authentication_features_10000.csv


In [ ]:
print("Rows:", len(dns_features))
print("Columns:", len(dns_features.columns))

Rows: 10000
Columns: 37


In [ ]:
structured_dns_path = os.path.join(
    external_dir,
    "dns_authentication_features_10000.csv"
)

dns_features.to_csv(
    structured_dns_path,
    index=False
)

print("Saved:")
print(structured_dns_path)

Saved:
/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/external/dns_authentication_features_10000.csv


In [ ]:
# ============================================
# DKIM OBSERVATION SCANNER
# ============================================

import pandas as pd
import dns.resolver
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
import time
import os

# Common/provider-associated selectors.
# These are observation candidates, NOT proof of DKIM failure.
DKIM_SELECTORS = [
    "google",
    "selector1",
    "selector2",
    "s1",
    "s2",
    "default",
    "k1",
    "k2"
]

domains = (
    dns_features["domain"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
    .drop_duplicates()
    .tolist()
)

print("Domains:", len(domains))
print("Selectors tested:", DKIM_SELECTORS)


def query_dkim_selector(domain, selector):

    resolver = dns.resolver.Resolver()
    resolver.timeout = 3
    resolver.lifetime = 5

    hostname = f"{selector}._domainkey.{domain}"

    # ----------------------------------------
    # First try TXT
    # ----------------------------------------
    try:
        answers = resolver.resolve(
            hostname,
            "TXT"
        )

        txt_records = []

        for answer in answers:
            text = answer.to_text()
            text = text.replace('" "', '')
            text = text.strip('"').strip()

            txt_records.append(text)

        dkim_records = [
            record
            for record in txt_records
            if (
                "p=" in record.lower()
                and (
                    "v=dkim1" in record.lower()
                    or "k=" in record.lower()
                )
            )
        ]

        if dkim_records:

            return {
                "selector": selector,
                "status": "FOUND_TXT",
                "record": " | ".join(dkim_records),
                "error": None
            }

    except (
        dns.resolver.NoAnswer,
        dns.resolver.NXDOMAIN,
        dns.resolver.NoNameservers,
        dns.exception.Timeout
    ):
        pass

    except Exception as e:

        return {
            "selector": selector,
            "status": "ERROR",
            "record": None,
            "error": type(e).__name__
        }

    # ----------------------------------------
    # Then try CNAME
    # ----------------------------------------
    try:
        answers = resolver.resolve(
            hostname,
            "CNAME"
        )

        cname_records = [
            str(answer.target).rstrip(".")
            for answer in answers
        ]

        if cname_records:

            return {
                "selector": selector,
                "status": "FOUND_CNAME",
                "record": " | ".join(cname_records),
                "error": None
            }

    except (
        dns.resolver.NoAnswer,
        dns.resolver.NXDOMAIN,
        dns.resolver.NoNameservers,
        dns.exception.Timeout
    ):
        pass

    except Exception as e:

        return {
            "selector": selector,
            "status": "ERROR",
            "record": None,
            "error": type(e).__name__
        }

    return {
        "selector": selector,
        "status": "NOT_FOUND",
        "record": None,
        "error": None
    }


def scan_dkim_domain(domain):

    checked_at = datetime.now(
        timezone.utc
    ).isoformat()

    observations = []

    for selector in DKIM_SELECTORS:

        result = query_dkim_selector(
            domain,
            selector
        )

        observations.append({
            "domain": domain,
            "selector": selector,
            "checked_at_utc": checked_at,
            "status": result["status"],
            "record": result["record"],
            "error": result["error"]
        })

    return observations


# ============================================
# Scan domains
# ============================================

dkim_results = []

start_time = time.time()

with ThreadPoolExecutor(
    max_workers=10
) as executor:

    futures = {
        executor.submit(
            scan_dkim_domain,
            domain
        ): domain
        for domain in domains
    }

    completed = 0

    for future in as_completed(futures):

        domain = futures[future]

        try:
            dkim_results.extend(
                future.result()
            )

        except Exception as e:

            for selector in DKIM_SELECTORS:

                dkim_results.append({
                    "domain": domain,
                    "selector": selector,
                    "checked_at_utc": datetime.now(
                        timezone.utc
                    ).isoformat(),
                    "status": "ERROR",
                    "record": None,
                    "error": str(e)
                })

        completed += 1

        if completed % 500 == 0:

            elapsed = time.time() - start_time

            print(
                f"Completed: {completed}/{len(domains)} "
                f"| {elapsed:.1f}s"
            )


dkim_observations = pd.DataFrame(
    dkim_results
)

dkim_observations = (
    dkim_observations
    .sort_values(["domain", "selector"])
    .reset_index(drop=True)
)

print("\nDKIM scan completed.")
print("Observation rows:", len(dkim_observations))

display(dkim_observations.head(30))

Domains: 10000
Selectors tested: ['google', 'selector1', 'selector2', 's1', 's2', 'default', 'k1', 'k2']
Completed: 500/10000 | 144.4s
Completed: 1000/10000 | 267.2s
Completed: 1500/10000 | 416.4s
Completed: 2000/10000 | 586.3s
Completed: 2500/10000 | 767.4s
Completed: 3000/10000 | 917.7s
Completed: 3500/10000 | 1062.6s
Completed: 4000/10000 | 1180.8s
Completed: 4500/10000 | 1348.1s
Completed: 5000/10000 | 1508.6s
Completed: 5500/10000 | 1684.4s
Completed: 6000/10000 | 1838.5s
Completed: 6500/10000 | 1989.0s
Completed: 7000/10000 | 2144.3s
Completed: 7500/10000 | 2310.2s
Completed: 8000/10000 | 2456.9s
Completed: 8500/10000 | 2618.8s
Completed: 9000/10000 | 2775.8s
Completed: 9500/10000 | 2919.1s
Completed: 10000/10000 | 3061.6s

DKIM scan completed.
Observation rows: 80000


,domain,selector,checked_at_utc,status,record,error
0,0123tt.ru,default,2026-09-22T08:52:14.979098+00:00,NOT_FOUND,None,None
1,0123tt.ru,google,2026-09-22T08:52:14.979098+00:00,NOT_FOUND,None,None
2,0123tt.ru,k1,2026-09-22T08:52:14.979098+00:00,NOT_FOUND,None,None
3,0123tt.ru,k2,2026-09-22T08:52:14.979098+00:00,NOT_FOUND,None,None
4,0123tt.ru,s1,2026-09-22T08:52:14.979098+00:00,NOT_FOUND,None,None
5,0123tt.ru,s2,2026-09-22T08:52:14.979098+00:00,NOT_FOUND,None,None
6,0123tt.ru,selector1,2026-09-22T08:52:14.979098+00:00,NOT_FOUND,None,None
7,0123tt.ru,selector2,2026-09-22T08:52:14.979098+00:00,NOT_FOUND,None,None
8,0lin.com,default,2026-09-22T08:52:14.982030+00:00,NOT_FOUND,None,None
9,0lin.com,google,2026-09-22T08:52:14.982030+00:00,NOT_FOUND,None,None


In [ ]:
print("Total observations:",
      len(dkim_observations))

print("\nStatus distribution:")
print(
    dkim_observations["status"]
    .value_counts()
)

print("\nDomains with at least one DKIM observation:")
print(
    dkim_observations.loc[
        dkim_observations["status"].isin(
            ["FOUND_TXT", "FOUND_CNAME"]
        ),
        "domain"
    ].nunique()
)

print("\nSelectors with observations:")
print(
    dkim_observations.loc[
        dkim_observations["status"].isin(
            ["FOUND_TXT", "FOUND_CNAME"]
        ),
        "selector"
    ].value_counts()
)

Total observations: 80000

Status distribution:
status
NOT_FOUND      64428
FOUND_TXT       8977
FOUND_CNAME     6595
Name: count, dtype: int64

Domains with at least one DKIM observation:
4753

Selectors with observations:
selector
s1           2561
google       2531
s2           2503
selector1    2052
selector2    2005
k1           1368
k2           1347
default      1205
Name: count, dtype: int64


In [ ]:
dkim_path = os.path.join(
    external_dir,
    "dkim_observations_10000.csv"
)

dkim_observations.to_csv(
    dkim_path,
    index=False
)

print("Saved:")
print(dkim_path)

Saved:
/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/external/dkim_observations_10000.csv


In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/AI_Email_Deliverability_Intelligence"
EXTERNAL_DIR = os.path.join(BASE_DIR, "data", "external")

print("Drive connected:", os.path.exists("/content/drive/MyDrive"))
print("Project folder exists:", os.path.exists(BASE_DIR))
print("External folder exists:", os.path.exists(EXTERNAL_DIR))

Drive connected: True
Project folder exists: True
External folder exists: True


In [ ]:
for filename in [
    "domain_population_10000.csv",
    "dns_observations_10000.csv",
    "dns_authentication_features_10000.csv"
]:

    path = os.path.join(EXTERNAL_DIR, filename)

    print(
        filename,
        "→",
        "AVAILABLE" if os.path.exists(path) else "MISSING"
    )

domain_population_10000.csv → AVAILABLE
dns_observations_10000.csv → AVAILABLE
dns_authentication_features_10000.csv → AVAILABLE


In [ ]:
print("dns_features:", dns_features.shape)
print("domain_population:", domain_population.shape)

dns_features: (10000, 37)
domain_population: (10000, 4)


In [ ]:
DKIM_SELECTORS = [
    "google",
    "selector1",
    "selector2",
    "s1",
    "s2",
    "default",
    "k1",
    "k2"
]

pilot_domains = (
    domain_population["domain"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
    .drop_duplicates()
    .head(1000)
    .tolist()
)

print("Pilot domains:", len(pilot_domains))
print("Selectors being tested:", DKIM_SELECTORS)

Pilot domains: 1000
Selectors being tested: ['google', 'selector1', 'selector2', 's1', 's2', 'default', 'k1', 'k2']


In [ ]:
# ============================================
# DKIM PILOT SCANNER
# ============================================

import dns.resolver
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
import time

def clean_txt_record(answer):
    text = answer.to_text()
    text = text.replace('" "', '')
    return text.strip('"').strip()


def query_dkim(domain, selector):

    resolver = dns.resolver.Resolver()
    resolver.timeout = 3
    resolver.lifetime = 5

    hostname = f"{selector}._domainkey.{domain}"

    # ----------------------------------------
    # TXT lookup
    # ----------------------------------------
    try:
        answers = resolver.resolve(
            hostname,
            "TXT"
        )

        txt_records = [
            clean_txt_record(answer)
            for answer in answers
        ]

        dkim_records = [
            record
            for record in txt_records
            if record.lower().startswith("v=dkim1")
        ]

        if dkim_records:
            return {
                "status": "FOUND_TXT",
                "record": " | ".join(dkim_records),
                "error": None
            }

    except dns.resolver.NXDOMAIN:
        return {
            "status": "NOT_FOUND",
            "record": None,
            "error": "NXDOMAIN"
        }

    except dns.resolver.NoAnswer:
        pass

    except dns.exception.Timeout:
        return {
            "status": "ERROR",
            "record": None,
            "error": "Timeout"
        }

    except Exception as e:
        return {
            "status": "ERROR",
            "record": None,
            "error": type(e).__name__
        }

    # ----------------------------------------
    # CNAME lookup
    # ----------------------------------------
    try:
        answers = resolver.resolve(
            hostname,
            "CNAME"
        )

        cname_records = [
            str(answer.target).rstrip(".")
            for answer in answers
        ]

        if cname_records:
            return {
                "status": "FOUND_CNAME",
                "record": " | ".join(cname_records),
                "error": None
            }

    except dns.resolver.NXDOMAIN:
        pass

    except dns.resolver.NoAnswer:
        pass

    except dns.exception.Timeout:
        return {
            "status": "ERROR",
            "record": None,
            "error": "Timeout"
        }

    except Exception as e:
        return {
            "status": "ERROR",
            "record": None,
            "error": type(e).__name__
        }

    return {
        "status": "NOT_FOUND",
        "record": None,
        "error": None
    }


def scan_dkim_domain(domain):

    observations = []

    checked_at = datetime.now(
        timezone.utc
    ).isoformat()

    for selector in DKIM_SELECTORS:

        result = query_dkim(
            domain,
            selector
        )

        observations.append({
            "domain": domain,
            "selector": selector,
            "checked_at_utc": checked_at,
            "status": result["status"],
            "record": result["record"],
            "error": result["error"]
        })

    return observations


# ============================================
# RUN PILOT
# ============================================

dkim_pilot_results = []

start_time = time.time()

with ThreadPoolExecutor(max_workers=5) as executor:

    futures = {
        executor.submit(
            scan_dkim_domain,
            domain
        ): domain
        for domain in pilot_domains
    }

    completed = 0

    for future in as_completed(futures):

        domain = futures[future]

        try:
            dkim_pilot_results.extend(
                future.result()
            )

        except Exception as e:

            for selector in DKIM_SELECTORS:

                dkim_pilot_results.append({
                    "domain": domain,
                    "selector": selector,
                    "checked_at_utc": datetime.now(
                        timezone.utc
                    ).isoformat(),
                    "status": "ERROR",
                    "record": None,
                    "error": str(e)
                })

        completed += 1

        if completed % 100 == 0:
            elapsed = time.time() - start_time

            print(
                f"Completed: {completed}/{len(pilot_domains)} "
                f"| Elapsed: {elapsed:.1f}s"
            )


dkim_pilot = pd.DataFrame(
    dkim_pilot_results
)

dkim_pilot = (
    dkim_pilot
    .sort_values(["domain", "selector"])
    .reset_index(drop=True)
)

print("\nDKIM pilot completed.")
print("Observation rows:", len(dkim_pilot))

display(dkim_pilot.head(20))

Completed: 100/1000 | Elapsed: 25.7s
Completed: 200/1000 | Elapsed: 60.6s
Completed: 300/1000 | Elapsed: 110.8s
Completed: 400/1000 | Elapsed: 162.3s
Completed: 500/1000 | Elapsed: 210.2s
Completed: 600/1000 | Elapsed: 259.3s
Completed: 700/1000 | Elapsed: 321.9s
Completed: 800/1000 | Elapsed: 377.4s
Completed: 900/1000 | Elapsed: 433.9s
Completed: 1000/1000 | Elapsed: 518.9s

DKIM pilot completed.
Observation rows: 8000


,domain,selector,checked_at_utc,status,record,error
0,163.com,default,2026-09-22T09:46:07.350381+00:00,NOT_FOUND,None,NXDOMAIN
1,163.com,google,2026-09-22T09:46:07.350381+00:00,NOT_FOUND,None,NXDOMAIN
2,163.com,k1,2026-09-22T09:46:07.350381+00:00,NOT_FOUND,None,NXDOMAIN
3,163.com,k2,2026-09-22T09:46:07.350381+00:00,NOT_FOUND,None,NXDOMAIN
4,163.com,s1,2026-09-22T09:46:07.350381+00:00,NOT_FOUND,None,NXDOMAIN
5,163.com,s2,2026-09-22T09:46:07.350381+00:00,NOT_FOUND,None,NXDOMAIN
6,163.com,selector1,2026-09-22T09:46:07.350381+00:00,NOT_FOUND,None,NXDOMAIN
7,163.com,selector2,2026-09-22T09:46:07.350381+00:00,NOT_FOUND,None,NXDOMAIN
8,1rx.io,default,2026-09-22T09:49:00.257411+00:00,NOT_FOUND,None,NXDOMAIN
9,1rx.io,google,2026-09-22T09:49:00.257411+00:00,NOT_FOUND,None,NXDOMAIN


In [ ]:
print("Status distribution:")
print(
    dkim_pilot["status"]
    .value_counts(dropna=False)
)

print(
    "\nDomains with at least one DKIM observation:",
    dkim_pilot.loc[
        dkim_pilot["status"].isin(
            ["FOUND_TXT", "FOUND_CNAME"]
        ),
        "domain"
    ].nunique()
)

print("\nSelectors with observations:")
print(
    dkim_pilot.loc[
        dkim_pilot["status"].isin(
            ["FOUND_TXT", "FOUND_CNAME"]
        ),
        "selector"
    ].value_counts()
)

Status distribution:
status
NOT_FOUND      6412
FOUND_TXT       619
FOUND_CNAME     542
ERROR           427
Name: count, dtype: int64

Domains with at least one DKIM observation: 429

Selectors with observations:
selector
google       272
s2           214
k2           151
k1           150
selector1    126
default       95
selector2     79
s1            74
Name: count, dtype: int64


In [ ]:
display(
    dkim_pilot[
        dkim_pilot["status"] == "FOUND_TXT"
    ][
        [
            "domain",
            "selector",
            "status",
            "record"
        ]
    ].head(20)
)

,domain,selector,status,record
33,33across.com,google,FOUND_TXT,v=DKIM1; k=rsa; p=MIIBIjANBgkqhkiG9w0BAQEFAAOC...
121,aboutads.info,google,FOUND_TXT,v=DKIM1;k=rsa;p=MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ...
136,academia.edu,default,FOUND_TXT,v=DKIM1; g=*; k=rsa; p=MIGfMA0GCSqGSIb3DQEBAQU...
137,academia.edu,google,FOUND_TXT,v=DKIM1; k=rsa; p=MIGfMA0GCSqGSIb3DQEBAQUAA4GN...
150,accuweather.com,selector1,FOUND_TXT,v=DKIM1; k=rsa; p=MIGfMA0GCSqGSIb3DQEBAQUAA4GN...
171,actify.nl,k2,FOUND_TXT,v=DKIM1; k=rsa; p=MIIBIjANBgkqhkiG9w0BAQEFAAOC...
185,addtoany.com,google,FOUND_TXT,v=DKIM1; k=rsa; p=MIIBIjANBgkqhkiG9w0BAQEFAAOC...
208,adjust.com,default,FOUND_TXT,v=DKIM1; h=sha256; k=rsa; p=MIIBIjANBgkqhkiG9w...
209,adjust.com,google,FOUND_TXT,v=DKIM1; k=rsa; p=MIIBIjANBgkqhkiG9w0BAQEFAAOC...
211,adjust.com,k2,FOUND_TXT,v=DKIM1; k=rsa; p=MIIBIjANBgkqhkiG9w0BAQEFAAOC...


In [ ]:
print("DKIM error distribution:")
print(
    dkim_pilot.loc[
        dkim_pilot["status"] == "ERROR",
        "error"
    ].value_counts(dropna=False)
)

DKIM error distribution:
error
Timeout          405
NoNameservers     22
Name: count, dtype: int64


In [ ]:
print("Errors by selector:")
print(
    dkim_pilot.loc[
        dkim_pilot["status"] == "ERROR",
        "selector"
    ].value_counts()
)

Errors by selector:
selector
s1           208
selector1     74
s2            63
selector2     59
default        6
google         6
k2             6
k1             5
Name: count, dtype: int64


In [ ]:
print("Errors by domain:")
print(
    dkim_pilot.loc[
        dkim_pilot["status"] == "ERROR",
        "domain"
    ].value_counts().head(20)
)

Errors by domain:
domain
abovedomains.com    8
b-cdn.net           8
ameblo.jp           8
list-manage.com     8
shiabank.com        6
fandom.com          5
sagepub.com         4
hilton.com          4
criteo.com          3
roku.com            3
princeton.edu       3
wix.com             3
bluehost.com        3
autodesk.com        3
trendyol.com        3
wiley.com           3
mlb.com             3
optimizely.com      3
hostgator.com.br    3
ieee.org            3
Name: count, dtype: int64


In [ ]:
found_dkim = dkim_pilot[
    dkim_pilot["status"].isin(
        ["FOUND_TXT", "FOUND_CNAME"]
    )
].copy()

print("Found DKIM observations:", len(found_dkim))
print(
    "Domains with observations:",
    found_dkim["domain"].nunique()
)

print("\nTXT observations:")
print(
    (found_dkim["status"] == "FOUND_TXT").sum()
)

print("\nCNAME observations:")
print(
    (found_dkim["status"] == "FOUND_CNAME").sum()
)

Found DKIM observations: 1161
Domains with observations: 429

TXT observations:
619

CNAME observations:
542


In [ ]:
txt_records = found_dkim[
    found_dkim["status"] == "FOUND_TXT"
].copy()

txt_records["has_dkim_version"] = (
    txt_records["record"]
    .fillna("")
    .str.contains(
        r"\bv=DKIM1\b",
        case=False,
        regex=True
    )
)

txt_records["has_public_key"] = (
    txt_records["record"]
    .fillna("")
    .str.contains(
        r"(?:^|;)\s*p=",
        case=False,
        regex=True
    )
)

print(
    "TXT records containing v=DKIM1:",
    txt_records["has_dkim_version"].sum()
)

print(
    "TXT records containing p=:",
    txt_records["has_public_key"].sum()
)

display(
    txt_records[
        [
            "domain",
            "selector",
            "record"
        ]
    ].head(20)
)

TXT records containing v=DKIM1: 619
TXT records containing p=: 619


,domain,selector,record
33,33across.com,google,v=DKIM1; k=rsa; p=MIIBIjANBgkqhkiG9w0BAQEFAAOC...
121,aboutads.info,google,v=DKIM1;k=rsa;p=MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ...
136,academia.edu,default,v=DKIM1; g=*; k=rsa; p=MIGfMA0GCSqGSIb3DQEBAQU...
137,academia.edu,google,v=DKIM1; k=rsa; p=MIGfMA0GCSqGSIb3DQEBAQUAA4GN...
150,accuweather.com,selector1,v=DKIM1; k=rsa; p=MIGfMA0GCSqGSIb3DQEBAQUAA4GN...
171,actify.nl,k2,v=DKIM1; k=rsa; p=MIIBIjANBgkqhkiG9w0BAQEFAAOC...
185,addtoany.com,google,v=DKIM1; k=rsa; p=MIIBIjANBgkqhkiG9w0BAQEFAAOC...
208,adjust.com,default,v=DKIM1; h=sha256; k=rsa; p=MIIBIjANBgkqhkiG9w...
209,adjust.com,google,v=DKIM1; k=rsa; p=MIIBIjANBgkqhkiG9w0BAQEFAAOC...
211,adjust.com,k2,v=DKIM1; k=rsa; p=MIIBIjANBgkqhkiG9w0BAQEFAAOC...


In [ ]:
print(
    dkim_pilot[
        dkim_pilot["status"] == "FOUND_CNAME"
    ][
        [
            "domain",
            "selector",
            "record"
        ]
    ].head(20)
)

              domain   selector  \
37      33across.com         s2   
138     academia.edu         k1   
139     academia.edu         k2   
140     academia.edu         s1   
141     academia.edu         s2   
142     academia.edu  selector1   
143     academia.edu  selector2   
146  accuweather.com         k1   
149  accuweather.com         s2   
157        achmea.nl         s2   
173        actify.nl         s2   
226        adobe.com         k1   
228        adobe.com         s1   
229        adobe.com         s2   
316        agoda.com         s1   
317        agoda.com         s2   
432   aliexpress.com    default   
433   aliexpress.com     google   
434   aliexpress.com         k1   
435   aliexpress.com         k2   

                                                record  
37             s2.domainkey.u395383.wl134.sendgrid.net  
138                                      dkim.mcsv.net  
139                    academia.edu.cdn.cloudflare.net  
140                    academia.edu.

In [ ]:
# ============================================
# RETRY FAILED DKIM QUERIES
# ============================================

failed_pairs = (
    dkim_pilot[
        dkim_pilot["status"] == "ERROR"
    ][
        ["domain", "selector"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Failed DKIM observations to retry:", len(failed_pairs))

Failed DKIM observations to retry: 427


In [ ]:
import dns.resolver
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
import time


def retry_dkim_query(domain, selector):

    resolver = dns.resolver.Resolver()

    # Longer timeout for previously failed queries
    resolver.timeout = 5
    resolver.lifetime = 12

    hostname = f"{selector}._domainkey.{domain}"

    # ----------------------------------------
    # TXT
    # ----------------------------------------

    try:

        answers = resolver.resolve(
            hostname,
            "TXT"
        )

        records = []

        for answer in answers:
            text = answer.to_text()
            text = text.replace('" "', '')
            text = text.strip('"').strip()
            records.append(text)

        dkim_records = [
            record
            for record in records
            if record.lower().startswith("v=dkim1")
        ]

        if dkim_records:

            return {
                "domain": domain,
                "selector": selector,
                "status": "FOUND_TXT",
                "record": " | ".join(dkim_records),
                "error": None
            }

    except (
        dns.resolver.NoAnswer,
        dns.resolver.NXDOMAIN
    ):
        pass

    except dns.exception.Timeout:

        return {
            "domain": domain,
            "selector": selector,
            "status": "ERROR",
            "record": None,
            "error": "Timeout"
        }

    except dns.resolver.NoNameservers:

        return {
            "domain": domain,
            "selector": selector,
            "status": "ERROR",
            "record": None,
            "error": "NoNameservers"
        }

    except Exception as e:

        return {
            "domain": domain,
            "selector": selector,
            "status": "ERROR",
            "record": None,
            "error": type(e).__name__
        }

    # ----------------------------------------
    # CNAME
    # ----------------------------------------

    try:

        answers = resolver.resolve(
            hostname,
            "CNAME"
        )

        cname_records = [
            str(answer.target).rstrip(".")
            for answer in answers
        ]

        if cname_records:

            return {
                "domain": domain,
                "selector": selector,
                "status": "FOUND_CNAME",
                "record": " | ".join(cname_records),
                "error": None
            }

    except (
        dns.resolver.NoAnswer,
        dns.resolver.NXDOMAIN
    ):
        pass

    except dns.exception.Timeout:

        return {
            "domain": domain,
            "selector": selector,
            "status": "ERROR",
            "record": None,
            "error": "Timeout"
        }

    except dns.resolver.NoNameservers:

        return {
            "domain": domain,
            "selector": selector,
            "status": "ERROR",
            "record": None,
            "error": "NoNameservers"
        }

    except Exception as e:

        return {
            "domain": domain,
            "selector": selector,
            "status": "ERROR",
            "record": None,
            "error": type(e).__name__
        }

    return {
        "domain": domain,
        "selector": selector,
        "status": "NOT_FOUND",
        "record": None,
        "error": None
    }


retry_results = []

start_time = time.time()

with ThreadPoolExecutor(max_workers=3) as executor:

    futures = {
        executor.submit(
            retry_dkim_query,
            row["domain"],
            row["selector"]
        ): (
            row["domain"],
            row["selector"]
        )
        for _, row in failed_pairs.iterrows()
    }

    completed = 0

    for future in as_completed(futures):

        result = future.result()

        result["checked_at_utc"] = (
            datetime.now(timezone.utc).isoformat()
        )

        retry_results.append(result)

        completed += 1

        if completed % 50 == 0:

            print(
                f"Retried: {completed}/{len(futures)}"
            )


dkim_retry = pd.DataFrame(retry_results)

print("\nRetry completed.")
print("Rows:", len(dkim_retry))

Retried: 50/427
Retried: 100/427
Retried: 150/427
Retried: 200/427
Retried: 250/427
Retried: 300/427
Retried: 350/427
Retried: 400/427

Retry completed.
Rows: 427


In [ ]:
print("Retry status distribution:")
print(
    dkim_retry["status"]
    .value_counts(dropna=False)
)

Retry status distribution:
status
ERROR          425
FOUND_CNAME      2
Name: count, dtype: int64


In [ ]:
print("Retry error distribution:")
print(
    dkim_retry.loc[
        dkim_retry["status"] == "ERROR",
        "error"
    ].value_counts(dropna=False)
)

Retry error distribution:
error
NoNameservers    425
Name: count, dtype: int64


In [ ]:
# ============================================
# TEST FAILED DKIM QUERIES WITH PUBLIC RESOLVERS
# ============================================

import dns.resolver
import pandas as pd

failed_sample = (
    dkim_retry[
        dkim_retry["status"] == "ERROR"
    ][
        ["domain", "selector"]
    ]
    .drop_duplicates()
    .head(20)
)

print("Testing failed observations:", len(failed_sample))


def test_with_resolver(domain, selector, nameserver):

    resolver = dns.resolver.Resolver(
        configure=False
    )

    resolver.nameservers = [nameserver]
    resolver.timeout = 4
    resolver.lifetime = 8

    hostname = f"{selector}._domainkey.{domain}"

    # TXT
    try:
        answers = resolver.resolve(
            hostname,
            "TXT"
        )

        records = [
            answer.to_text()
            for answer in answers
        ]

        return {
            "resolver": nameserver,
            "status": "TXT_FOUND",
            "record": " | ".join(records),
            "error": None
        }

    except dns.resolver.NoAnswer:
        pass

    except dns.resolver.NXDOMAIN:
        return {
            "resolver": nameserver,
            "status": "NOT_FOUND",
            "record": None,
            "error": "NXDOMAIN"
        }

    except dns.resolver.NoNameservers as e:
        return {
            "resolver": nameserver,
            "status": "ERROR",
            "record": None,
            "error": "NoNameservers"
        }

    except dns.exception.Timeout:
        return {
            "resolver": nameserver,
            "status": "ERROR",
            "record": None,
            "error": "Timeout"
        }

    except Exception as e:
        return {
            "resolver": nameserver,
            "status": "ERROR",
            "record": None,
            "error": type(e).__name__
        }

    # CNAME
    try:
        answers = resolver.resolve(
            hostname,
            "CNAME"
        )

        records = [
            str(answer.target).rstrip(".")
            for answer in answers
        ]

        if records:
            return {
                "resolver": nameserver,
                "status": "CNAME_FOUND",
                "record": " | ".join(records),
                "error": None
            }

    except dns.resolver.NXDOMAIN:
        return {
            "resolver": nameserver,
            "status": "NOT_FOUND",
            "record": None,
            "error": "NXDOMAIN"
        }

    except dns.resolver.NoAnswer:
        return {
            "resolver": nameserver,
            "status": "NOT_FOUND",
            "record": None,
            "error": "NoAnswer"
        }

    except Exception as e:
        return {
            "resolver": nameserver,
            "status": "ERROR",
            "record": None,
            "error": type(e).__name__
        }

    return {
        "resolver": nameserver,
        "status": "NOT_FOUND",
        "record": None,
        "error": None
    }


resolver_test_results = []

for _, row in failed_sample.iterrows():

    domain = row["domain"]
    selector = row["selector"]

    for nameserver in [
        "1.1.1.1",
        "8.8.8.8"
    ]:

        result = test_with_resolver(
            domain,
            selector,
            nameserver
        )

        resolver_test_results.append({
            "domain": domain,
            "selector": selector,
            **result
        })


resolver_test = pd.DataFrame(
    resolver_test_results
)

display(resolver_test)

Testing failed observations: 20


,domain,selector,resolver,status,record,error
0,abovedomains.com,default,1.1.1.1,NOT_FOUND,None,NoAnswer
1,abovedomains.com,default,8.8.8.8,ERROR,None,NoNameservers
2,abovedomains.com,google,1.1.1.1,NOT_FOUND,None,NoAnswer
3,abovedomains.com,google,8.8.8.8,ERROR,None,NoNameservers
4,abovedomains.com,k2,1.1.1.1,NOT_FOUND,None,NoAnswer
5,abovedomains.com,k2,8.8.8.8,ERROR,None,NoNameservers
6,abovedomains.com,k1,1.1.1.1,NOT_FOUND,None,NoAnswer
7,abovedomains.com,k1,8.8.8.8,ERROR,None,NoNameservers
8,abovedomains.com,s1,1.1.1.1,NOT_FOUND,None,NoAnswer
9,abovedomains.com,s1,8.8.8.8,ERROR,None,NoNameservers


In [ ]:
print("\nResolver test summary:")
print(
    resolver_test
    .groupby("resolver")["status"]
    .value_counts()
)


Resolver test summary:
resolver  status   
1.1.1.1   TXT_FOUND    12
          NOT_FOUND     8
8.8.8.8   TXT_FOUND    12
          ERROR         8
Name: count, dtype: int64


In [ ]:
print("\nComparison:")
display(
    resolver_test[
        [
            "domain",
            "selector",
            "resolver",
            "status",
            "error"
        ]
    ]
)


Comparison:


,domain,selector,resolver,status,error
0,abovedomains.com,default,1.1.1.1,NOT_FOUND,NoAnswer
1,abovedomains.com,default,8.8.8.8,ERROR,NoNameservers
2,abovedomains.com,google,1.1.1.1,NOT_FOUND,NoAnswer
3,abovedomains.com,google,8.8.8.8,ERROR,NoNameservers
4,abovedomains.com,k2,1.1.1.1,NOT_FOUND,NoAnswer
5,abovedomains.com,k2,8.8.8.8,ERROR,NoNameservers
6,abovedomains.com,k1,1.1.1.1,NOT_FOUND,NoAnswer
7,abovedomains.com,k1,8.8.8.8,ERROR,NoNameservers
8,abovedomains.com,s1,1.1.1.1,NOT_FOUND,NoAnswer
9,abovedomains.com,s1,8.8.8.8,ERROR,NoNameservers


In [ ]:
# ============================================
# DUAL-RESOLVER DKIM PILOT
# ============================================

import dns.resolver
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
import time

DKIM_RESOLVERS = {
    "cloudflare": "1.1.1.1",
    "google": "8.8.8.8"
}

# Same 1,000-domain pilot
pilot_domains = (
    domain_population["domain"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
    .drop_duplicates()
    .head(1000)
    .tolist()
)

print("Pilot domains:", len(pilot_domains))
print("Resolvers:", DKIM_RESOLVERS)
print("Selectors:", DKIM_SELECTORS)

Pilot domains: 1000
Resolvers: {'cloudflare': '1.1.1.1', 'google': '8.8.8.8'}
Selectors: ['google', 'selector1', 'selector2', 's1', 's2', 'default', 'k1', 'k2']


In [ ]:
# ============================================
# DNS HELPER
# ============================================

def clean_txt_record(answer):
    text = answer.to_text()
    text = text.replace('" "', '')
    return text.strip('"').strip()


def query_dkim_resolver(domain, selector, nameserver):

    resolver = dns.resolver.Resolver(
        configure=False
    )

    resolver.nameservers = [nameserver]

    resolver.timeout = 4
    resolver.lifetime = 8

    hostname = f"{selector}._domainkey.{domain}"

    # ----------------------------------------
    # TXT
    # ----------------------------------------

    try:

        answers = resolver.resolve(
            hostname,
            "TXT"
        )

        records = [
            clean_txt_record(answer)
            for answer in answers
        ]

        dkim_records = [
            record
            for record in records
            if record.lower().startswith("v=dkim1")
        ]

        if dkim_records:

            return {
                "status": "FOUND_TXT",
                "record": " | ".join(
                    dkim_records
                ),
                "error": None
            }

    except dns.resolver.NXDOMAIN:

        return {
            "status": "NOT_FOUND",
            "record": None,
            "error": "NXDOMAIN"
        }

    except dns.resolver.NoAnswer:
        pass

    except dns.resolver.NoNameservers:

        return {
            "status": "ERROR",
            "record": None,
            "error": "NoNameservers"
        }

    except dns.exception.Timeout:

        return {
            "status": "ERROR",
            "record": None,
            "error": "Timeout"
        }

    except Exception as e:

        return {
            "status": "ERROR",
            "record": None,
            "error": type(e).__name__
        }

    # ----------------------------------------
    # CNAME
    # ----------------------------------------

    try:

        answers = resolver.resolve(
            hostname,
            "CNAME"
        )

        cname_records = [
            str(answer.target).rstrip(".")
            for answer in answers
        ]

        if cname_records:

            return {
                "status": "FOUND_CNAME",
                "record": " | ".join(
                    cname_records
                ),
                "error": None
            }

    except dns.resolver.NXDOMAIN:

        return {
            "status": "NOT_FOUND",
            "record": None,
            "error": "NXDOMAIN"
        }

    except dns.resolver.NoAnswer:

        return {
            "status": "NOT_FOUND",
            "record": None,
            "error": "NoAnswer"
        }

    except dns.resolver.NoNameservers:

        return {
            "status": "ERROR",
            "record": None,
            "error": "NoNameservers"
        }

    except dns.exception.Timeout:

        return {
            "status": "ERROR",
            "record": None,
            "error": "Timeout"
        }

    except Exception as e:

        return {
            "status": "ERROR",
            "record": None,
            "error": type(e).__name__
        }

    return {
        "status": "NOT_FOUND",
        "record": None,
        "error": None
    }


# ============================================
# QUERY BOTH RESOLVERS
# ============================================

def scan_dkim_selector(domain, selector):

    observations = []

    for resolver_name, nameserver in DKIM_RESOLVERS.items():

        result = query_dkim_resolver(
            domain,
            selector,
            nameserver
        )

        observations.append({
            "domain": domain,
            "selector": selector,
            "resolver": resolver_name,
            "nameserver": nameserver,
            "checked_at_utc":
                datetime.now(timezone.utc).isoformat(),
            "status": result["status"],
            "record": result["record"],
            "error": result["error"]
        })

    return observations


# ============================================
# RUN PILOT
# ============================================

resolver_results = []

start_time = time.time()

# Small worker count to avoid excessive DNS pressure
with ThreadPoolExecutor(max_workers=5) as executor:

    futures = {
        executor.submit(
            scan_dkim_selector,
            domain,
            selector
        ): (domain, selector)

        for domain in pilot_domains
        for selector in DKIM_SELECTORS
    }

    completed = 0

    for future in as_completed(futures):

        try:

            resolver_results.extend(
                future.result()
            )

        except Exception as e:

            domain, selector = futures[future]

            for resolver_name, nameserver in DKIM_RESOLVERS.items():

                resolver_results.append({
                    "domain": domain,
                    "selector": selector,
                    "resolver": resolver_name,
                    "nameserver": nameserver,
                    "checked_at_utc":
                        datetime.now(timezone.utc).isoformat(),
                    "status": "ERROR",
                    "record": None,
                    "error": str(e)
                })

        completed += 1

        if completed % 500 == 0:

            print(
                f"Completed selector pairs: "
                f"{completed}/{len(futures)}"
            )


dkim_dual_pilot = pd.DataFrame(
    resolver_results
)

dkim_dual_pilot = (
    dkim_dual_pilot
    .sort_values(
        ["domain", "selector", "resolver"]
    )
    .reset_index(drop=True)
)

print("\nDual-resolver pilot completed.")
print("Observation rows:", len(dkim_dual_pilot))

display(dkim_dual_pilot.head(20))

Completed selector pairs: 500/8000
Completed selector pairs: 1000/8000
Completed selector pairs: 1500/8000
Completed selector pairs: 2000/8000
Completed selector pairs: 2500/8000
Completed selector pairs: 3000/8000
Completed selector pairs: 3500/8000
Completed selector pairs: 4000/8000
Completed selector pairs: 4500/8000
Completed selector pairs: 5000/8000
Completed selector pairs: 5500/8000
Completed selector pairs: 6000/8000
Completed selector pairs: 6500/8000
Completed selector pairs: 7000/8000
Completed selector pairs: 7500/8000
Completed selector pairs: 8000/8000

Dual-resolver pilot completed.
Observation rows: 16000


,domain,selector,resolver,nameserver,checked_at_utc,status,record,error
0,163.com,default,cloudflare,1.1.1.1,2026-09-22T10:13:46.573076+00:00,NOT_FOUND,None,NXDOMAIN
1,163.com,default,google,8.8.8.8,2026-09-22T10:13:46.619448+00:00,NOT_FOUND,None,NXDOMAIN
2,163.com,google,cloudflare,1.1.1.1,2026-09-22T10:13:46.168198+00:00,NOT_FOUND,None,NXDOMAIN
3,163.com,google,google,8.8.8.8,2026-09-22T10:13:46.374829+00:00,NOT_FOUND,None,NXDOMAIN
4,163.com,k1,cloudflare,1.1.1.1,2026-09-22T10:13:46.688265+00:00,NOT_FOUND,None,NXDOMAIN
5,163.com,k1,google,8.8.8.8,2026-09-22T10:13:46.734614+00:00,NOT_FOUND,None,NXDOMAIN
6,163.com,k2,cloudflare,1.1.1.1,2026-09-22T10:13:46.686247+00:00,NOT_FOUND,None,NXDOMAIN
7,163.com,k2,google,8.8.8.8,2026-09-22T10:13:46.732135+00:00,NOT_FOUND,None,NXDOMAIN
8,163.com,s1,cloudflare,1.1.1.1,2026-09-22T10:13:46.470140+00:00,NOT_FOUND,None,NXDOMAIN
9,163.com,s1,google,8.8.8.8,2026-09-22T10:13:46.695656+00:00,NOT_FOUND,None,NXDOMAIN


In [ ]:
# ============================================
# FINAL STATUS PER DOMAIN + SELECTOR
# ============================================

def combine_resolver_results(group):

    statuses = set(
        group["status"]
    )

    # ----------------------------------------
    # Any successful observation wins
    # ----------------------------------------

    if "FOUND_TXT" in statuses:

        row = group[
            group["status"] == "FOUND_TXT"
        ].iloc[0]

        return pd.Series({
            "final_status": "FOUND_TXT",
            "selected_record": row["record"],
            "selected_resolver": row["resolver"],
            "resolver_agreement": True
        })

    if "FOUND_CNAME" in statuses:

        row = group[
            group["status"] == "FOUND_CNAME"
        ].iloc[0]

        return pd.Series({
            "final_status": "FOUND_CNAME",
            "selected_record": row["record"],
            "selected_resolver": row["resolver"],
            "resolver_agreement": True
        })

    # ----------------------------------------
    # Both resolvers definitively say absent
    # ----------------------------------------

    if statuses.issubset({"NOT_FOUND"}):

        return pd.Series({
            "final_status": "NOT_FOUND",
            "selected_record": None,
            "selected_resolver": None,
            "resolver_agreement": True
        })

    # ----------------------------------------
    # Otherwise unresolved
    # ----------------------------------------

    return pd.Series({
        "final_status": "UNRESOLVED",
        "selected_record": None,
        "selected_resolver": None,
        "resolver_agreement": False
    })


dkim_selector_final = (
    dkim_dual_pilot
    .groupby(
        ["domain", "selector"],
        as_index=False
    )
    .apply(
        combine_resolver_results,
        include_groups=False
    )
    .reset_index()
)

# Remove unnecessary groupby index column if present
if "level_2" in dkim_selector_final.columns:
    dkim_selector_final = dkim_selector_final.drop(
        columns=["level_2"]
    )

display(
    dkim_selector_final.head(30)
)

,index,domain,selector,final_status,selected_record,selected_resolver,resolver_agreement
0,0,163.com,default,NOT_FOUND,None,None,True
1,1,163.com,google,NOT_FOUND,None,None,True
2,2,163.com,k1,NOT_FOUND,None,None,True
3,3,163.com,k2,NOT_FOUND,None,None,True
4,4,163.com,s1,NOT_FOUND,None,None,True
5,5,163.com,s2,NOT_FOUND,None,None,True
6,6,163.com,selector1,NOT_FOUND,None,None,True
7,7,163.com,selector2,NOT_FOUND,None,None,True
8,8,1rx.io,default,NOT_FOUND,None,None,True
9,9,1rx.io,google,NOT_FOUND,None,None,True


In [ ]:
print("Final DKIM selector status:")
print(
    dkim_selector_final[
        "final_status"
    ].value_counts()
)

Final DKIM selector status:
final_status
NOT_FOUND      6408
FOUND_CNAME     830
FOUND_TXT       744
UNRESOLVED       18
Name: count, dtype: int64


In [ ]:
print(
    "Domains with at least one DKIM observation:",
    dkim_selector_final.loc[
        dkim_selector_final["final_status"].isin(
            ["FOUND_TXT", "FOUND_CNAME"]
        ),
        "domain"
    ].nunique()
)

Domains with at least one DKIM observation: 465


In [ ]:
print("\nResolver disagreement:")
print(
    (
        dkim_dual_pilot
        .groupby(
            ["domain", "selector"]
        )["status"]
        .nunique()
        .gt(1)
        .sum()
    )
)


Resolver disagreement:
25


In [ ]:
# ============================================
# INSPECT RESOLVER DISAGREEMENTS
# ============================================

resolver_comparison = (
    dkim_dual_pilot
    .groupby(
        ["domain", "selector"]
    )
    .agg(
        resolver_count=("resolver", "nunique"),
        status_count=("status", "nunique"),
        statuses=("status", lambda x: " | ".join(sorted(set(x)))),
        errors=("error", lambda x: " | ".join(
            sorted(set(
                str(v) for v in x.dropna()
            ))
        )),
        records=("record", lambda x: " | ".join(
            sorted(set(
                str(v) for v in x.dropna()
            ))
        ))
    )
    .reset_index()
)

disagreements = resolver_comparison[
    resolver_comparison["status_count"] > 1
]

print("Resolver disagreements:", len(disagreements))

display(disagreements)

Resolver disagreements: 25


,domain,selector,resolver_count,status_count,statuses,errors,records
128,abovedomains.com,default,2,2,ERROR | NOT_FOUND,NoAnswer | NoNameservers,
129,abovedomains.com,google,2,2,ERROR | NOT_FOUND,NoAnswer | NoNameservers,
130,abovedomains.com,k1,2,2,ERROR | NOT_FOUND,NoAnswer | NoNameservers,
131,abovedomains.com,k2,2,2,ERROR | NOT_FOUND,NoAnswer | NoNameservers,
132,abovedomains.com,s1,2,2,ERROR | NOT_FOUND,NoAnswer | NoNameservers,
133,abovedomains.com,s2,2,2,ERROR | NOT_FOUND,NoAnswer | NoNameservers,
134,abovedomains.com,selector1,2,2,ERROR | NOT_FOUND,NoAnswer | NoNameservers,
135,abovedomains.com,selector2,2,2,ERROR | NOT_FOUND,NoAnswer | NoNameservers,
1024,b-cdn.net,default,2,2,ERROR | NOT_FOUND,NoAnswer | NoNameservers,
1025,b-cdn.net,google,2,2,ERROR | NOT_FOUND,NoAnswer | NoNameservers,


In [ ]:
print(
    disagreements[
        [
            "statuses",
            "errors"
        ]
    ].value_counts()
)

statuses                 errors                  
ERROR | NOT_FOUND        NoAnswer | NoNameservers    16
ERROR | FOUND_CNAME      NoNameservers                5
FOUND_CNAME | NOT_FOUND  NXDOMAIN                     4
Name: count, dtype: int64


In [ ]:
# ============================================
# FINALIZE DKIM PILOT
# ============================================

def final_dkim_status(group):

    statuses = set(group["status"])

    # Successful TXT observation
    if "FOUND_TXT" in statuses:
        return "FOUND_TXT"

    # Successful CNAME observation
    if "FOUND_CNAME" in statuses:
        return "FOUND_CNAME"

    # Both resolvers definitively say absent
    if statuses == {"NOT_FOUND"}:
        return "NOT_FOUND"

    # Anything involving an unresolved DNS result
    return "UNRESOLVED"


dkim_final_status = (
    dkim_dual_pilot
    .groupby(
        ["domain", "selector"]
    )
    .apply(
        final_dkim_status,
        include_groups=False
    )
    .rename("final_status")
    .reset_index()
)

display(dkim_final_status.head(20))

,domain,selector,final_status
0,163.com,default,NOT_FOUND
1,163.com,google,NOT_FOUND
2,163.com,k1,NOT_FOUND
3,163.com,k2,NOT_FOUND
4,163.com,s1,NOT_FOUND
5,163.com,s2,NOT_FOUND
6,163.com,selector1,NOT_FOUND
7,163.com,selector2,NOT_FOUND
8,1rx.io,default,NOT_FOUND
9,1rx.io,google,NOT_FOUND


In [ ]:
resolver_summary = (
    dkim_dual_pilot
    .groupby(
        ["domain", "selector"]
    )
    .agg(
        resolver_statuses=(
            "status",
            lambda x: " | ".join(sorted(set(x)))
        ),
        resolver_errors=(
            "error",
            lambda x: " | ".join(
                sorted(
                    set(
                        str(v)
                        for v in x.dropna()
                    )
                )
            )
        ),
        observation_count=(
            "status",
            "count"
        )
    )
    .reset_index()
)

dkim_final = dkim_final_status.merge(
    resolver_summary,
    on=["domain", "selector"],
    how="left"
)

print("Final DKIM pilot:")
display(dkim_final.head(20))

Final DKIM pilot:


,domain,selector,final_status,resolver_statuses,resolver_errors,observation_count
0,163.com,default,NOT_FOUND,NOT_FOUND,NXDOMAIN,2
1,163.com,google,NOT_FOUND,NOT_FOUND,NXDOMAIN,2
2,163.com,k1,NOT_FOUND,NOT_FOUND,NXDOMAIN,2
3,163.com,k2,NOT_FOUND,NOT_FOUND,NXDOMAIN,2
4,163.com,s1,NOT_FOUND,NOT_FOUND,NXDOMAIN,2
5,163.com,s2,NOT_FOUND,NOT_FOUND,NXDOMAIN,2
6,163.com,selector1,NOT_FOUND,NOT_FOUND,NXDOMAIN,2
7,163.com,selector2,NOT_FOUND,NOT_FOUND,NXDOMAIN,2
8,1rx.io,default,NOT_FOUND,NOT_FOUND,NXDOMAIN,2
9,1rx.io,google,NOT_FOUND,NOT_FOUND,NXDOMAIN,2


In [ ]:
print("Final status distribution:")

print(
    dkim_final["final_status"]
    .value_counts()
)

print(
    "\nDomains with at least one successful DKIM observation:",
    dkim_final.loc[
        dkim_final["final_status"].isin(
            ["FOUND_TXT", "FOUND_CNAME"]
        ),
        "domain"
    ].nunique()
)

print(
    "\nUnresolved observations:",
    (
        dkim_final["final_status"] == "UNRESOLVED"
    ).sum()
)

Final status distribution:
final_status
NOT_FOUND      6408
FOUND_CNAME     830
FOUND_TXT       744
UNRESOLVED       18
Name: count, dtype: int64

Domains with at least one successful DKIM observation: 465

Unresolved observations: 18


In [ ]:
dkim_pilot_final_path = os.path.join(
    external_dir,
    "dkim_observations_pilot_1000.csv"
)

dkim_final.to_csv(
    dkim_pilot_final_path,
    index=False
)

print("Saved:")
print(dkim_pilot_final_path)

Saved:
/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/external/dkim_observations_pilot_1000.csv


In [ ]:
# ============================================
# DKIM FULL-SCAN BATCH 2
# Domains 1,001–2,000
# ============================================

batch_number = 2

batch_domains = (
    domain_population["domain"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
    .drop_duplicates()
    .iloc[1000:2000]
    .tolist()
)

print("Batch number:", batch_number)
print("Domains in batch:", len(batch_domains))

Batch number: 2
Domains in batch: 1000


In [ ]:
# ============================================
# SCAN BATCH
# ============================================

batch_results = []

start_time = time.time()

with ThreadPoolExecutor(max_workers=5) as executor:

    futures = {
        executor.submit(
            scan_dkim_selector,
            domain,
            selector
        ): (domain, selector)

        for domain in batch_domains
        for selector in DKIM_SELECTORS
    }

    completed = 0

    for future in as_completed(futures):

        try:
            batch_results.extend(
                future.result()
            )

        except Exception as e:

            domain, selector = futures[future]

            for resolver_name, nameserver in DKIM_RESOLVERS.items():

                batch_results.append({
                    "domain": domain,
                    "selector": selector,
                    "resolver": resolver_name,
                    "nameserver": nameserver,
                    "checked_at_utc":
                        datetime.now(timezone.utc).isoformat(),
                    "status": "ERROR",
                    "record": None,
                    "error": str(e)
                })

        completed += 1

        if completed % 500 == 0:
            print(
                f"Completed: {completed}/"
                f"{len(futures)} selector pairs"
            )


dkim_batch_raw = pd.DataFrame(batch_results)

print("\nBatch scan completed.")
print("Raw observations:", len(dkim_batch_raw))

Completed: 500/8000 selector pairs
Completed: 1000/8000 selector pairs
Completed: 1500/8000 selector pairs
Completed: 2000/8000 selector pairs
Completed: 2500/8000 selector pairs
Completed: 3000/8000 selector pairs
Completed: 3500/8000 selector pairs
Completed: 4000/8000 selector pairs
Completed: 4500/8000 selector pairs
Completed: 5000/8000 selector pairs
Completed: 5500/8000 selector pairs
Completed: 6000/8000 selector pairs
Completed: 6500/8000 selector pairs
Completed: 7000/8000 selector pairs
Completed: 7500/8000 selector pairs
Completed: 8000/8000 selector pairs

Batch scan completed.
Raw observations: 16000


In [ ]:
# ============================================
# FINALIZE BATCH STATUS
# ============================================

batch_status = (
    dkim_batch_raw
    .groupby(
        ["domain", "selector"]
    )
    .apply(
        final_dkim_status,
        include_groups=False
    )
    .rename("final_status")
    .reset_index()
)

resolver_summary = (
    dkim_batch_raw
    .groupby(
        ["domain", "selector"]
    )
    .agg(
        resolver_statuses=(
            "status",
            lambda x: " | ".join(
                sorted(set(x))
            )
        ),
        resolver_errors=(
            "error",
            lambda x: " | ".join(
                sorted(
                    set(
                        str(v)
                        for v in x.dropna()
                    )
                )
            )
        )
    )
    .reset_index()
)

dkim_batch_final = batch_status.merge(
    resolver_summary,
    on=["domain", "selector"],
    how="left"
)

print(
    dkim_batch_final["final_status"]
    .value_counts()
)

final_status
NOT_FOUND      6599
FOUND_CNAME     716
FOUND_TXT       627
UNRESOLVED       58
Name: count, dtype: int64


In [ ]:
# ============================================
# SAVE BATCH
# ============================================

batch_path = os.path.join(
    external_dir,
    f"dkim_observations_batch_{batch_number}.csv"
)

dkim_batch_final.to_csv(
    batch_path,
    index=False
)

print("Batch saved:")
print(batch_path)

Batch saved:
/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/external/dkim_observations_batch_2.csv


In [ ]:
print(
    "Domains:",
    dkim_batch_final["domain"].nunique()
)

print(
    "Observations:",
    len(dkim_batch_final)
)

Domains: 1000
Observations: 8000


In [ ]:
# ============================================
# DKIM FULL-SCAN BATCH 3
# Domains 2,001–3,000
# ============================================

batch_number = 3

batch_domains = (
    domain_population["domain"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
    .drop_duplicates()
    .iloc[2000:3000]
    .tolist()
)

print("Batch number:", batch_number)
print("Domains in batch:", len(batch_domains))

Batch number: 3
Domains in batch: 1000


In [ ]:
# ============================================
# SCAN BATCH 3
# ============================================

batch_results = []

start_time = time.time()

with ThreadPoolExecutor(max_workers=5) as executor:

    futures = {
        executor.submit(
            scan_dkim_selector,
            domain,
            selector
        ): (domain, selector)

        for domain in batch_domains
        for selector in DKIM_SELECTORS
    }

    completed = 0

    for future in as_completed(futures):

        try:
            batch_results.extend(
                future.result()
            )

        except Exception as e:

            domain, selector = futures[future]

            for resolver_name, nameserver in DKIM_RESOLVERS.items():

                batch_results.append({
                    "domain": domain,
                    "selector": selector,
                    "resolver": resolver_name,
                    "nameserver": nameserver,
                    "checked_at_utc":
                        datetime.now(timezone.utc).isoformat(),
                    "status": "ERROR",
                    "record": None,
                    "error": str(e)
                })

        completed += 1

        if completed % 500 == 0:
            print(
                f"Completed: {completed}/"
                f"{len(futures)} selector pairs"
            )

dkim_batch_3_raw = pd.DataFrame(batch_results)

print("\nBatch 3 scan completed.")
print("Raw observations:", len(dkim_batch_3_raw))

Completed: 500/8000 selector pairs
Completed: 1000/8000 selector pairs
Completed: 1500/8000 selector pairs
Completed: 2000/8000 selector pairs
Completed: 2500/8000 selector pairs
Completed: 3000/8000 selector pairs
Completed: 3500/8000 selector pairs
Completed: 4000/8000 selector pairs
Completed: 4500/8000 selector pairs
Completed: 5000/8000 selector pairs
Completed: 5500/8000 selector pairs
Completed: 6000/8000 selector pairs
Completed: 6500/8000 selector pairs
Completed: 7000/8000 selector pairs
Completed: 7500/8000 selector pairs
Completed: 8000/8000 selector pairs

Batch 3 scan completed.
Raw observations: 16000


In [ ]:
# ============================================
# FINALIZE BATCH 3
# ============================================

batch_status = (
    dkim_batch_3_raw
    .groupby(["domain", "selector"])
    .apply(
        final_dkim_status,
        include_groups=False
    )
    .rename("final_status")
    .reset_index()
)

resolver_summary = (
    dkim_batch_3_raw
    .groupby(["domain", "selector"])
    .agg(
        resolver_statuses=(
            "status",
            lambda x: " | ".join(sorted(set(x)))
        ),
        resolver_errors=(
            "error",
            lambda x: " | ".join(
                sorted(
                    set(
                        str(v)
                        for v in x.dropna()
                    )
                )
            )
        )
    )
    .reset_index()
)

dkim_batch_3_final = batch_status.merge(
    resolver_summary,
    on=["domain", "selector"],
    how="left"
)

print("Batch 3 status:")
print(
    dkim_batch_3_final["final_status"]
    .value_counts()
)

Batch 3 status:
final_status
NOT_FOUND      6581
FOUND_CNAME     703
FOUND_TXT       663
UNRESOLVED       53
Name: count, dtype: int64


In [ ]:
print(
    "Domains:",
    dkim_batch_3_final["domain"].nunique()
)

print(
    "Observations:",
    len(dkim_batch_3_final)
)

Domains: 1000
Observations: 8000


In [ ]:
batch_3_path = os.path.join(
    external_dir,
    "dkim_observations_batch_3.csv"
)

dkim_batch_3_final.to_csv(
    batch_3_path,
    index=False
)

print("Batch 3 saved:")
print(batch_3_path)

Batch 3 saved:
/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/external/dkim_observations_batch_3.csv


In [ ]:
# ============================================
# DKIM FULL-SCAN BATCH 4
# Domains 3,001–4,000
# ============================================

batch_number = 4

batch_domains = (
    domain_population["domain"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
    .drop_duplicates()
    .iloc[3000:4000]
    .tolist()
)

print("Batch number:", batch_number)
print("Domains in batch:", len(batch_domains))

Batch number: 4
Domains in batch: 1000


In [ ]:
# ============================================
# SCAN BATCH 4
# ============================================

batch_results = []

start_time = time.time()

with ThreadPoolExecutor(max_workers=5) as executor:

    futures = {
        executor.submit(
            scan_dkim_selector,
            domain,
            selector
        ): (domain, selector)

        for domain in batch_domains
        for selector in DKIM_SELECTORS
    }

    completed = 0

    for future in as_completed(futures):

        try:
            batch_results.extend(
                future.result()
            )

        except Exception as e:

            domain, selector = futures[future]

            for resolver_name, nameserver in DKIM_RESOLVERS.items():

                batch_results.append({
                    "domain": domain,
                    "selector": selector,
                    "resolver": resolver_name,
                    "nameserver": nameserver,
                    "checked_at_utc":
                        datetime.now(timezone.utc).isoformat(),
                    "status": "ERROR",
                    "record": None,
                    "error": str(e)
                })

        completed += 1

        if completed % 500 == 0:
            print(
                f"Completed: {completed}/"
                f"{len(futures)} selector pairs"
            )

dkim_batch_4_raw = pd.DataFrame(batch_results)

print("\nBatch 4 scan completed.")
print("Raw observations:", len(dkim_batch_4_raw))

Completed: 500/8000 selector pairs
Completed: 1000/8000 selector pairs
Completed: 1500/8000 selector pairs
Completed: 2000/8000 selector pairs
Completed: 2500/8000 selector pairs
Completed: 3000/8000 selector pairs
Completed: 3500/8000 selector pairs
Completed: 4000/8000 selector pairs
Completed: 4500/8000 selector pairs
Completed: 5000/8000 selector pairs
Completed: 5500/8000 selector pairs
Completed: 6000/8000 selector pairs
Completed: 6500/8000 selector pairs
Completed: 7000/8000 selector pairs
Completed: 7500/8000 selector pairs
Completed: 8000/8000 selector pairs

Batch 4 scan completed.
Raw observations: 16000


In [ ]:
# ============================================
# FINALIZE BATCH 4
# ============================================

batch_status = (
    dkim_batch_4_raw
    .groupby(["domain", "selector"])
    .apply(
        final_dkim_status,
        include_groups=False
    )
    .rename("final_status")
    .reset_index()
)

resolver_summary = (
    dkim_batch_4_raw
    .groupby(["domain", "selector"])
    .agg(
        resolver_statuses=(
            "status",
            lambda x: " | ".join(sorted(set(x)))
        ),
        resolver_errors=(
            "error",
            lambda x: " | ".join(
                sorted(
                    set(
                        str(v)
                        for v in x.dropna()
                    )
                )
            )
        )
    )
    .reset_index()
)

dkim_batch_4_final = batch_status.merge(
    resolver_summary,
    on=["domain", "selector"],
    how="left"
)

print("Batch 4 status:")
print(
    dkim_batch_4_final["final_status"]
    .value_counts()
)

Batch 4 status:
final_status
NOT_FOUND      6169
FOUND_TXT      1102
FOUND_CNAME     708
UNRESOLVED       21
Name: count, dtype: int64


In [ ]:
print(
    "Domains:",
    dkim_batch_4_final["domain"].nunique()
)

print(
    "Observations:",
    len(dkim_batch_4_final)
)

Domains: 1000
Observations: 8000


In [ ]:
batch_4_path = os.path.join(
    external_dir,
    "dkim_observations_batch_4.csv"
)

dkim_batch_4_final.to_csv(
    batch_4_path,
    index=False
)

print("Batch 4 saved:")
print(batch_4_path)

Batch 4 saved:
/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/external/dkim_observations_batch_4.csv


In [ ]:
# ============================================
# DKIM FULL-SCAN BATCH 5
# Domains 4,001–5,000
# ============================================

batch_number = 5

batch_domains = (
    domain_population["domain"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
    .drop_duplicates()
    .iloc[4000:5000]
    .tolist()
)

print("Batch number:", batch_number)
print("Domains in batch:", len(batch_domains))

Batch number: 5
Domains in batch: 1000


In [ ]:
# ============================================
# SCAN BATCH 5
# ============================================

batch_results = []

start_time = time.time()

with ThreadPoolExecutor(max_workers=5) as executor:

    futures = {
        executor.submit(
            scan_dkim_selector,
            domain,
            selector
        ): (domain, selector)

        for domain in batch_domains
        for selector in DKIM_SELECTORS
    }

    completed = 0

    for future in as_completed(futures):

        try:
            batch_results.extend(
                future.result()
            )

        except Exception as e:

            domain, selector = futures[future]

            for resolver_name, nameserver in DKIM_RESOLVERS.items():

                batch_results.append({
                    "domain": domain,
                    "selector": selector,
                    "resolver": resolver_name,
                    "nameserver": nameserver,
                    "checked_at_utc":
                        datetime.now(timezone.utc).isoformat(),
                    "status": "ERROR",
                    "record": None,
                    "error": str(e)
                })

        completed += 1

        if completed % 500 == 0:
            print(
                f"Completed: {completed}/"
                f"{len(futures)} selector pairs"
            )

dkim_batch_5_raw = pd.DataFrame(batch_results)

print("\nBatch 5 scan completed.")
print("Raw observations:", len(dkim_batch_5_raw))

Completed: 500/8000 selector pairs
Completed: 1000/8000 selector pairs
Completed: 1500/8000 selector pairs
Completed: 2000/8000 selector pairs
Completed: 2500/8000 selector pairs
Completed: 3000/8000 selector pairs
Completed: 3500/8000 selector pairs
Completed: 4000/8000 selector pairs
Completed: 4500/8000 selector pairs
Completed: 5000/8000 selector pairs
Completed: 5500/8000 selector pairs
Completed: 6000/8000 selector pairs
Completed: 6500/8000 selector pairs
Completed: 7000/8000 selector pairs
Completed: 7500/8000 selector pairs
Completed: 8000/8000 selector pairs

Batch 5 scan completed.
Raw observations: 16000


In [ ]:
# ============================================
# FINALIZE BATCH 5
# ============================================

batch_status = (
    dkim_batch_5_raw
    .groupby(["domain", "selector"])
    .apply(
        final_dkim_status,
        include_groups=False
    )
    .rename("final_status")
    .reset_index()
)

resolver_summary = (
    dkim_batch_5_raw
    .groupby(["domain", "selector"])
    .agg(
        resolver_statuses=(
            "status",
            lambda x: " | ".join(sorted(set(x)))
        ),
        resolver_errors=(
            "error",
            lambda x: " | ".join(
                sorted(
                    set(
                        str(v)
                        for v in x.dropna()
                    )
                )
            )
        )
    )
    .reset_index()
)

dkim_batch_5_final = batch_status.merge(
    resolver_summary,
    on=["domain", "selector"],
    how="left"
)

print("Batch 5 status:")
print(
    dkim_batch_5_final["final_status"]
    .value_counts()
)

Batch 5 status:
final_status
NOT_FOUND      6501
FOUND_TXT       743
FOUND_CNAME     724
UNRESOLVED       32
Name: count, dtype: int64


In [ ]:
print(
    "Domains:",
    dkim_batch_5_final["domain"].nunique()
)

print(
    "Observations:",
    len(dkim_batch_5_final)
)

Domains: 1000
Observations: 8000


In [ ]:
batch_5_path = os.path.join(
    external_dir,
    "dkim_observations_batch_5.csv"
)

dkim_batch_5_final.to_csv(
    batch_5_path,
    index=False
)

print("Batch 5 saved:")
print(batch_5_path)

Batch 5 saved:
/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/external/dkim_observations_batch_5.csv


In [ ]:
# ============================================
# DKIM FULL-SCAN BATCH 6
# Domains 5,001–6,000
# ============================================

batch_number = 6

batch_domains = (
    domain_population["domain"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
    .drop_duplicates()
    .iloc[5000:6000]
    .tolist()
)

print("Batch number:", batch_number)
print("Domains in batch:", len(batch_domains))

Batch number: 6
Domains in batch: 1000


In [ ]:
# ============================================
# SCAN BATCH 6
# ============================================

batch_results = []

start_time = time.time()

with ThreadPoolExecutor(max_workers=5) as executor:

    futures = {
        executor.submit(
            scan_dkim_selector,
            domain,
            selector
        ): (domain, selector)

        for domain in batch_domains
        for selector in DKIM_SELECTORS
    }

    completed = 0

    for future in as_completed(futures):

        try:
            batch_results.extend(
                future.result()
            )

        except Exception as e:

            domain, selector = futures[future]

            for resolver_name, nameserver in DKIM_RESOLVERS.items():

                batch_results.append({
                    "domain": domain,
                    "selector": selector,
                    "resolver": resolver_name,
                    "nameserver": nameserver,
                    "checked_at_utc":
                        datetime.now(timezone.utc).isoformat(),
                    "status": "ERROR",
                    "record": None,
                    "error": str(e)
                })

        completed += 1

        if completed % 500 == 0:
            print(
                f"Completed: {completed}/"
                f"{len(futures)} selector pairs"
            )

dkim_batch_6_raw = pd.DataFrame(batch_results)

print("\nBatch 6 scan completed.")
print("Raw observations:", len(dkim_batch_6_raw))

Completed: 500/8000 selector pairs
Completed: 1000/8000 selector pairs
Completed: 1500/8000 selector pairs
Completed: 2000/8000 selector pairs
Completed: 2500/8000 selector pairs
Completed: 3000/8000 selector pairs
Completed: 3500/8000 selector pairs
Completed: 4000/8000 selector pairs
Completed: 4500/8000 selector pairs
Completed: 5000/8000 selector pairs
Completed: 5500/8000 selector pairs
Completed: 6000/8000 selector pairs
Completed: 6500/8000 selector pairs
Completed: 7000/8000 selector pairs
Completed: 7500/8000 selector pairs
Completed: 8000/8000 selector pairs

Batch 6 scan completed.
Raw observations: 16000


In [ ]:
# ============================================
# FINALIZE BATCH 6
# ============================================

batch_status = (
    dkim_batch_6_raw
    .groupby(["domain", "selector"])
    .apply(
        final_dkim_status,
        include_groups=False
    )
    .rename("final_status")
    .reset_index()
)

resolver_summary = (
    dkim_batch_6_raw
    .groupby(["domain", "selector"])
    .agg(
        resolver_statuses=(
            "status",
            lambda x: " | ".join(sorted(set(x)))
        ),
        resolver_errors=(
            "error",
            lambda x: " | ".join(
                sorted(
                    set(
                        str(v)
                        for v in x.dropna()
                    )
                )
            )
        )
    )
    .reset_index()
)

dkim_batch_6_final = batch_status.merge(
    resolver_summary,
    on=["domain", "selector"],
    how="left"
)

print("Batch 6 status:")
print(
    dkim_batch_6_final["final_status"]
    .value_counts()
)

Batch 6 status:
final_status
NOT_FOUND      6505
FOUND_CNAME     781
FOUND_TXT       677
UNRESOLVED       37
Name: count, dtype: int64


In [ ]:
print(
    "Domains:",
    dkim_batch_6_final["domain"].nunique()
)

print(
    "Observations:",
    len(dkim_batch_6_final)
)

Domains: 1000
Observations: 8000


In [ ]:
batch_6_path = os.path.join(
    external_dir,
    "dkim_observations_batch_6.csv"
)

dkim_batch_6_final.to_csv(
    batch_6_path,
    index=False
)

print("Batch 6 saved:")
print(batch_6_path)

Batch 6 saved:
/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/external/dkim_observations_batch_6.csv


In [ ]:
# ============================================
# DKIM FULL-SCAN BATCH 7
# Domains 6,001–7,000
# ============================================

batch_number = 7

batch_domains = (
    domain_population["domain"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
    .drop_duplicates()
    .iloc[6000:7000]
    .tolist()
)

print("Batch number:", batch_number)
print("Domains in batch:", len(batch_domains))

Batch number: 7
Domains in batch: 1000


In [ ]:
# ============================================
# SCAN BATCH 7
# ============================================

batch_results = []

start_time = time.time()

with ThreadPoolExecutor(max_workers=5) as executor:

    futures = {
        executor.submit(
            scan_dkim_selector,
            domain,
            selector
        ): (domain, selector)

        for domain in batch_domains
        for selector in DKIM_SELECTORS
    }

    completed = 0

    for future in as_completed(futures):

        try:
            batch_results.extend(
                future.result()
            )

        except Exception as e:

            domain, selector = futures[future]

            for resolver_name, nameserver in DKIM_RESOLVERS.items():

                batch_results.append({
                    "domain": domain,
                    "selector": selector,
                    "resolver": resolver_name,
                    "nameserver": nameserver,
                    "checked_at_utc":
                        datetime.now(timezone.utc).isoformat(),
                    "status": "ERROR",
                    "record": None,
                    "error": str(e)
                })

        completed += 1

        if completed % 500 == 0:
            print(
                f"Completed: {completed}/"
                f"{len(futures)} selector pairs"
            )

dkim_batch_7_raw = pd.DataFrame(batch_results)

print("\nBatch 7 scan completed.")
print("Raw observations:", len(dkim_batch_7_raw))

Completed: 500/8000 selector pairs
Completed: 1000/8000 selector pairs
Completed: 1500/8000 selector pairs
Completed: 2000/8000 selector pairs
Completed: 2500/8000 selector pairs
Completed: 3000/8000 selector pairs
Completed: 3500/8000 selector pairs
Completed: 4000/8000 selector pairs
Completed: 4500/8000 selector pairs
Completed: 5000/8000 selector pairs
Completed: 5500/8000 selector pairs
Completed: 6000/8000 selector pairs
Completed: 6500/8000 selector pairs
Completed: 7000/8000 selector pairs
Completed: 7500/8000 selector pairs
Completed: 8000/8000 selector pairs

Batch 7 scan completed.
Raw observations: 16000


In [ ]:
# ============================================
# FINALIZE BATCH 7
# ============================================

batch_status = (
    dkim_batch_7_raw
    .groupby(["domain", "selector"])
    .apply(
        final_dkim_status,
        include_groups=False
    )
    .rename("final_status")
    .reset_index()
)

resolver_summary = (
    dkim_batch_7_raw
    .groupby(["domain", "selector"])
    .agg(
        resolver_statuses=(
            "status",
            lambda x: " | ".join(sorted(set(x)))
        ),
        resolver_errors=(
            "error",
            lambda x: " | ".join(
                sorted(
                    set(
                        str(v)
                        for v in x.dropna()
                    )
                )
            )
        )
    )
    .reset_index()
)

dkim_batch_7_final = batch_status.merge(
    resolver_summary,
    on=["domain", "selector"],
    how="left"
)

print("Batch 7 status:")
print(
    dkim_batch_7_final["final_status"]
    .value_counts()
)

Batch 7 status:
final_status
NOT_FOUND      6257
FOUND_TXT       988
FOUND_CNAME     710
UNRESOLVED       45
Name: count, dtype: int64


In [ ]:
print(
    "Domains:",
    dkim_batch_7_final["domain"].nunique()
)

print(
    "Observations:",
    len(dkim_batch_7_final)
)

Domains: 1000
Observations: 8000


In [ ]:
batch_7_path = os.path.join(
    external_dir,
    "dkim_observations_batch_7.csv"
)

dkim_batch_7_final.to_csv(
    batch_7_path,
    index=False
)

print("Batch 7 saved:")
print(batch_7_path)

Batch 7 saved:
/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/external/dkim_observations_batch_7.csv


In [ ]:
# ============================================
# DKIM FULL-SCAN BATCH 8
# Domains 7,001–8,000
# ============================================

batch_number = 8

batch_domains = (
    domain_population["domain"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
    .drop_duplicates()
    .iloc[7000:8000]
    .tolist()
)

print("Batch number:", batch_number)
print("Domains in batch:", len(batch_domains))

Batch number: 8
Domains in batch: 1000


In [ ]:
# ============================================
# SCAN BATCH 8
# ============================================

batch_results = []

start_time = time.time()

with ThreadPoolExecutor(max_workers=5) as executor:

    futures = {
        executor.submit(
            scan_dkim_selector,
            domain,
            selector
        ): (domain, selector)

        for domain in batch_domains
        for selector in DKIM_SELECTORS
    }

    completed = 0

    for future in as_completed(futures):

        try:
            batch_results.extend(
                future.result()
            )

        except Exception as e:

            domain, selector = futures[future]

            for resolver_name, nameserver in DKIM_RESOLVERS.items():

                batch_results.append({
                    "domain": domain,
                    "selector": selector,
                    "resolver": resolver_name,
                    "nameserver": nameserver,
                    "checked_at_utc":
                        datetime.now(timezone.utc).isoformat(),
                    "status": "ERROR",
                    "record": None,
                    "error": str(e)
                })

        completed += 1

        if completed % 500 == 0:
            print(
                f"Completed: {completed}/"
                f"{len(futures)} selector pairs"
            )

dkim_batch_8_raw = pd.DataFrame(batch_results)

print("\nBatch 8 scan completed.")
print("Raw observations:", len(dkim_batch_8_raw))

Completed: 500/8000 selector pairs
Completed: 1000/8000 selector pairs
Completed: 1500/8000 selector pairs
Completed: 2000/8000 selector pairs
Completed: 2500/8000 selector pairs
Completed: 3000/8000 selector pairs
Completed: 3500/8000 selector pairs
Completed: 4000/8000 selector pairs
Completed: 4500/8000 selector pairs
Completed: 5000/8000 selector pairs
Completed: 5500/8000 selector pairs
Completed: 6000/8000 selector pairs
Completed: 6500/8000 selector pairs
Completed: 7000/8000 selector pairs
Completed: 7500/8000 selector pairs
Completed: 8000/8000 selector pairs

Batch 8 scan completed.
Raw observations: 16000


In [ ]:
# ============================================
# FINALIZE BATCH 8
# ============================================

batch_status = (
    dkim_batch_8_raw
    .groupby(["domain", "selector"])
    .apply(
        final_dkim_status,
        include_groups=False
    )
    .rename("final_status")
    .reset_index()
)

resolver_summary = (
    dkim_batch_8_raw
    .groupby(["domain", "selector"])
    .agg(
        resolver_statuses=(
            "status",
            lambda x: " | ".join(sorted(set(x)))
        ),
        resolver_errors=(
            "error",
            lambda x: " | ".join(
                sorted(
                    set(
                        str(v)
                        for v in x.dropna()
                    )
                )
            )
        )
    )
    .reset_index()
)

dkim_batch_8_final = batch_status.merge(
    resolver_summary,
    on=["domain", "selector"],
    how="left"
)

print("Batch 8 status:")
print(
    dkim_batch_8_final["final_status"]
    .value_counts()
)

Batch 8 status:
final_status
NOT_FOUND      6257
FOUND_TXT      1030
FOUND_CNAME     659
UNRESOLVED       54
Name: count, dtype: int64


In [ ]:
print(
    "Domains:",
    dkim_batch_8_final["domain"].nunique()
)

print(
    "Observations:",
    len(dkim_batch_8_final)
)

Domains: 1000
Observations: 8000


In [ ]:
batch_8_path = os.path.join(
    external_dir,
    "dkim_observations_batch_8.csv"
)

dkim_batch_8_final.to_csv(
    batch_8_path,
    index=False
)

print("Batch 8 saved:")
print(batch_8_path)

Batch 8 saved:
/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/external/dkim_observations_batch_8.csv


In [ ]:
# ============================================
# DKIM FULL-SCAN BATCH 9
# Domains 8,001–9,000
# ============================================

batch_number = 9

batch_domains = (
    domain_population["domain"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
    .drop_duplicates()
    .iloc[8000:9000]
    .tolist()
)

print("Batch number:", batch_number)
print("Domains in batch:", len(batch_domains))

Batch number: 9
Domains in batch: 1000


In [ ]:
# ============================================
# SCAN BATCH 9
# ============================================

batch_results = []

start_time = time.time()

with ThreadPoolExecutor(max_workers=5) as executor:

    futures = {
        executor.submit(
            scan_dkim_selector,
            domain,
            selector
        ): (domain, selector)

        for domain in batch_domains
        for selector in DKIM_SELECTORS
    }

    completed = 0

    for future in as_completed(futures):

        try:
            batch_results.extend(
                future.result()
            )

        except Exception as e:

            domain, selector = futures[future]

            for resolver_name, nameserver in DKIM_RESOLVERS.items():

                batch_results.append({
                    "domain": domain,
                    "selector": selector,
                    "resolver": resolver_name,
                    "nameserver": nameserver,
                    "checked_at_utc":
                        datetime.now(timezone.utc).isoformat(),
                    "status": "ERROR",
                    "record": None,
                    "error": str(e)
                })

        completed += 1

        if completed % 500 == 0:
            print(
                f"Completed: {completed}/"
                f"{len(futures)} selector pairs"
            )

dkim_batch_9_raw = pd.DataFrame(batch_results)

print("\nBatch 9 scan completed.")
print("Raw observations:", len(dkim_batch_9_raw))

Completed: 500/8000 selector pairs
Completed: 1000/8000 selector pairs
Completed: 1500/8000 selector pairs
Completed: 2000/8000 selector pairs
Completed: 2500/8000 selector pairs
Completed: 3000/8000 selector pairs
Completed: 3500/8000 selector pairs
Completed: 4000/8000 selector pairs
Completed: 4500/8000 selector pairs
Completed: 5000/8000 selector pairs
Completed: 5500/8000 selector pairs
Completed: 6000/8000 selector pairs
Completed: 6500/8000 selector pairs
Completed: 7000/8000 selector pairs
Completed: 7500/8000 selector pairs
Completed: 8000/8000 selector pairs

Batch 9 scan completed.
Raw observations: 16000


In [ ]:
# ============================================
# FINALIZE BATCH 9
# ============================================

batch_status = (
    dkim_batch_9_raw
    .groupby(["domain", "selector"])
    .apply(
        final_dkim_status,
        include_groups=False
    )
    .rename("final_status")
    .reset_index()
)

resolver_summary = (
    dkim_batch_9_raw
    .groupby(["domain", "selector"])
    .agg(
        resolver_statuses=(
            "status",
            lambda x: " | ".join(sorted(set(x)))
        ),
        resolver_errors=(
            "error",
            lambda x: " | ".join(
                sorted(
                    set(
                        str(v)
                        for v in x.dropna()
                    )
                )
            )
        )
    )
    .reset_index()
)

dkim_batch_9_final = batch_status.merge(
    resolver_summary,
    on=["domain", "selector"],
    how="left"
)

print("Batch 9 status:")
print(
    dkim_batch_9_final["final_status"]
    .value_counts()
)

Batch 9 status:
final_status
NOT_FOUND      6754
FOUND_TXT       710
FOUND_CNAME     505
UNRESOLVED       31
Name: count, dtype: int64


In [ ]:
print(
    "Domains:",
    dkim_batch_9_final["domain"].nunique()
)

print(
    "Observations:",
    len(dkim_batch_9_final)
)

Domains: 1000
Observations: 8000


In [ ]:
batch_9_path = os.path.join(
    external_dir,
    "dkim_observations_batch_9.csv"
)

dkim_batch_9_final.to_csv(
    batch_9_path,
    index=False
)

print("Batch 9 saved:")
print(batch_9_path)

Batch 9 saved:
/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/external/dkim_observations_batch_9.csv


In [ ]:
# ============================================
# DKIM FULL-SCAN BATCH 10
# Domains 9,001–10,000
# ============================================

batch_number = 10

batch_domains = (
    domain_population["domain"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
    .drop_duplicates()
    .iloc[9000:10000]
    .tolist()
)

print("Batch number:", batch_number)
print("Domains in batch:", len(batch_domains))

Batch number: 10
Domains in batch: 1000


In [ ]:
# ============================================
# SCAN BATCH 10
# ============================================

batch_results = []

start_time = time.time()

with ThreadPoolExecutor(max_workers=5) as executor:

    futures = {
        executor.submit(
            scan_dkim_selector,
            domain,
            selector
        ): (domain, selector)

        for domain in batch_domains
        for selector in DKIM_SELECTORS
    }

    completed = 0

    for future in as_completed(futures):

        try:
            batch_results.extend(
                future.result()
            )

        except Exception as e:

            domain, selector = futures[future]

            for resolver_name, nameserver in DKIM_RESOLVERS.items():

                batch_results.append({
                    "domain": domain,
                    "selector": selector,
                    "resolver": resolver_name,
                    "nameserver": nameserver,
                    "checked_at_utc":
                        datetime.now(timezone.utc).isoformat(),
                    "status": "ERROR",
                    "record": None,
                    "error": str(e)
                })

        completed += 1

        if completed % 500 == 0:
            print(
                f"Completed: {completed}/"
                f"{len(futures)} selector pairs"
            )

dkim_batch_10_raw = pd.DataFrame(batch_results)

print("\nBatch 10 scan completed.")
print("Raw observations:", len(dkim_batch_10_raw))

Completed: 500/8000 selector pairs
Completed: 1000/8000 selector pairs
Completed: 1500/8000 selector pairs
Completed: 2000/8000 selector pairs
Completed: 2500/8000 selector pairs
Completed: 3000/8000 selector pairs
Completed: 3500/8000 selector pairs
Completed: 4000/8000 selector pairs
Completed: 4500/8000 selector pairs
Completed: 5000/8000 selector pairs
Completed: 5500/8000 selector pairs
Completed: 6000/8000 selector pairs
Completed: 6500/8000 selector pairs
Completed: 7000/8000 selector pairs
Completed: 7500/8000 selector pairs
Completed: 8000/8000 selector pairs

Batch 10 scan completed.
Raw observations: 16000


In [ ]:
# ============================================
# FINALIZE BATCH 10
# ============================================

batch_status = (
    dkim_batch_10_raw
    .groupby(["domain", "selector"])
    .apply(
        final_dkim_status,
        include_groups=False
    )
    .rename("final_status")
    .reset_index()
)

resolver_summary = (
    dkim_batch_10_raw
    .groupby(["domain", "selector"])
    .agg(
        resolver_statuses=(
            "status",
            lambda x: " | ".join(sorted(set(x)))
        ),
        resolver_errors=(
            "error",
            lambda x: " | ".join(
                sorted(
                    set(
                        str(v)
                        for v in x.dropna()
                    )
                )
            )
        )
    )
    .reset_index()
)

dkim_batch_10_final = batch_status.merge(
    resolver_summary,
    on=["domain", "selector"],
    how="left"
)

print("Batch 10 status:")
print(
    dkim_batch_10_final["final_status"]
    .value_counts()
)

Batch 10 status:
final_status
NOT_FOUND      7147
FOUND_TXT       518
FOUND_CNAME     307
UNRESOLVED       28
Name: count, dtype: int64


In [ ]:
print(
    "Domains:",
    dkim_batch_10_final["domain"].nunique()
)

print(
    "Observations:",
    len(dkim_batch_10_final)
)

Domains: 1000
Observations: 8000


In [ ]:
batch_10_path = os.path.join(
    external_dir,
    "dkim_observations_batch_10.csv"
)

dkim_batch_10_final.to_csv(
    batch_10_path,
    index=False
)

print("Batch 10 saved:")
print(batch_10_path)

Batch 10 saved:
/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/external/dkim_observations_batch_10.csv


In [ ]:
# ============================================
# COMBINE ALL DKIM BATCHES
# ============================================

import pandas as pd
import os

BASE_DIR = "/content/drive/MyDrive/AI_Email_Deliverability_Intelligence"
EXTERNAL_DIR = os.path.join(BASE_DIR, "data", "external")

dkim_files = []

# Batch 1 = validated 1,000-domain pilot
pilot_path = os.path.join(
    EXTERNAL_DIR,
    "dkim_observations_pilot_1000.csv"
)

if os.path.exists(pilot_path):
    dkim_files.append(pilot_path)

# Batches 2–10
for batch_number in range(2, 11):

    path = os.path.join(
        EXTERNAL_DIR,
        f"dkim_observations_batch_{batch_number}.csv"
    )

    if os.path.exists(path):
        dkim_files.append(path)

print("DKIM files found:", len(dkim_files))

for path in dkim_files:
    print(os.path.basename(path))

DKIM files found: 10
dkim_observations_pilot_1000.csv
dkim_observations_batch_2.csv
dkim_observations_batch_3.csv
dkim_observations_batch_4.csv
dkim_observations_batch_5.csv
dkim_observations_batch_6.csv
dkim_observations_batch_7.csv
dkim_observations_batch_8.csv
dkim_observations_batch_9.csv
dkim_observations_batch_10.csv


In [ ]:
# ============================================
# LOAD AND COMBINE
# ============================================

dkim_frames = []

for path in dkim_files:

    temp = pd.read_csv(path)

    temp["source_file"] = os.path.basename(path)

    dkim_frames.append(temp)


dkim_all = pd.concat(
    dkim_frames,
    ignore_index=True
)

print("Combined DKIM shape:", dkim_all.shape)

display(dkim_all.head())

Combined DKIM shape: (80000, 7)


,domain,selector,final_status,resolver_statuses,resolver_errors,observation_count,source_file
0,163.com,default,NOT_FOUND,NOT_FOUND,NXDOMAIN,2.0,dkim_observations_pilot_1000.csv
1,163.com,google,NOT_FOUND,NOT_FOUND,NXDOMAIN,2.0,dkim_observations_pilot_1000.csv
2,163.com,k1,NOT_FOUND,NOT_FOUND,NXDOMAIN,2.0,dkim_observations_pilot_1000.csv
3,163.com,k2,NOT_FOUND,NOT_FOUND,NXDOMAIN,2.0,dkim_observations_pilot_1000.csv
4,163.com,s1,NOT_FOUND,NOT_FOUND,NXDOMAIN,2.0,dkim_observations_pilot_1000.csv


In [ ]:
print("Total rows:", len(dkim_all))
print(
    "Unique domains:",
    dkim_all["domain"].nunique()
)
print(
    "Unique selectors:",
    dkim_all["selector"].nunique()
)

print("\nStatus distribution:")
print(
    dkim_all["final_status"]
    .value_counts(dropna=False)
)

Total rows: 80000
Unique domains: 10000
Unique selectors: 8

Status distribution:
final_status
NOT_FOUND      65178
FOUND_TXT       7802
FOUND_CNAME     6643
UNRESOLVED       377
Name: count, dtype: int64


In [ ]:
# Each domain should have 8 tested selectors

selector_counts = (
    dkim_all
    .groupby("domain")["selector"]
    .nunique()
)

print(
    "Domains with exactly 8 selectors:",
    (selector_counts == 8).sum()
)

print(
    "Domains with fewer than 8:",
    (selector_counts < 8).sum()
)

print(
    "Domains with more than 8:",
    (selector_counts > 8).sum()
)

Domains with exactly 8 selectors: 10000
Domains with fewer than 8: 0
Domains with more than 8: 0


In [ ]:
duplicate_check = dkim_all.duplicated(
    subset=["domain", "selector"]
).sum()

print(
    "Duplicate domain-selector pairs:",
    duplicate_check
)

Duplicate domain-selector pairs: 0


In [ ]:
# ============================================
# DOMAIN-LEVEL DKIM FEATURES
# ============================================

dkim_domain_features = (
    dkim_all
    .groupby("domain")
    .agg(
        dkim_selectors_tested=(
            "selector",
            "nunique"
        ),

        dkim_txt_count=(
            "final_status",
            lambda x: (x == "FOUND_TXT").sum()
        ),

        dkim_cname_count=(
            "final_status",
            lambda x: (x == "FOUND_CNAME").sum()
        ),

        dkim_not_found_count=(
            "final_status",
            lambda x: (x == "NOT_FOUND").sum()
        ),

        dkim_unresolved_count=(
            "final_status",
            lambda x: (x == "UNRESOLVED").sum()
        )
    )
    .reset_index()
)

dkim_domain_features["dkim_observation_found"] = (
    (
        dkim_domain_features["dkim_txt_count"]
        +
        dkim_domain_features["dkim_cname_count"]
    ) > 0
)

display(dkim_domain_features.head(20))

,domain,dkim_selectors_tested,dkim_txt_count,dkim_cname_count,dkim_not_found_count,dkim_unresolved_count,dkim_observation_found
0,0123tt.ru,8,0,0,8,0,False
1,0lin.com,8,0,0,8,0,False
2,0xrpc.io,8,0,0,8,0,False
3,10086.cn,8,0,0,8,0,False
4,1024tera.com,8,0,0,8,0,False
5,10jqka.com.cn,8,0,0,8,0,False
6,123-hdx.com,8,0,0,8,0,False
7,123av.com,8,0,0,8,0,False
8,123moviesfree.net,8,0,0,8,0,False
9,123rf.com,8,1,0,7,0,True


In [ ]:
# ============================================
# MERGE DNS + DKIM
# ============================================

external_features = dns_features.merge(
    dkim_domain_features,
    on="domain",
    how="left",
    validate="one_to_one"
)

print(
    "Final external feature shape:",
    external_features.shape
)

print(
    "Unique domains:",
    external_features["domain"].nunique()
)

Final external feature shape: (10000, 43)
Unique domains: 10000


In [ ]:
external_features = external_features.merge(
    domain_population[
        [
            "domain",
            "population_source",
            "population_rank",
            "collected_at_utc"
        ]
    ],
    on="domain",
    how="left",
    validate="one_to_one"
)

print(
    "Final shape:",
    external_features.shape
)

Final shape: (10000, 46)


In [ ]:
print("Domains:", external_features["domain"].nunique())

print(
    "MX available:",
    external_features["mx_present"].sum()
)

print(
    "SPF available:",
    external_features["spf_present"].sum()
)

print(
    "DMARC available:",
    external_features["dmarc_present"].sum()
)

print(
    "DKIM observed:",
    external_features["dkim_observation_found"].sum()
)

Domains: 10000
MX available: 7283
SPF available: 3749
DMARC available: 6705
DKIM observed: 4714


In [ ]:
dkim_all_path = os.path.join(
    EXTERNAL_DIR,
    "dkim_observations_10000.csv"
)

dkim_all.to_csv(
    dkim_all_path,
    index=False
)

print("Saved:", dkim_all_path)

Saved: /content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/external/dkim_observations_10000.csv


In [ ]:
dkim_domain_path = os.path.join(
    EXTERNAL_DIR,
    "dkim_domain_features_10000.csv"
)

dkim_domain_features.to_csv(
    dkim_domain_path,
    index=False
)

print("Saved:", dkim_domain_path)

Saved: /content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/external/dkim_domain_features_10000.csv


In [ ]:
external_features_path = os.path.join(
    EXTERNAL_DIR,
    "external_deliverability_features_10000.csv"
)

external_features.to_csv(
    external_features_path,
    index=False
)

print("Saved:", external_features_path)

Saved: /content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/external/external_deliverability_features_10000.csv


In [ ]:
# ============================================
# FINAL EXTERNAL DATASET VALIDATION
# ============================================

final_external = pd.read_csv(
    external_features_path
)

print("Rows:", len(final_external))
print("Columns:", len(final_external.columns))

print(
    "Unique domains:",
    final_external["domain"].nunique()
)

print(
    "Duplicate domains:",
    final_external["domain"].duplicated().sum()
)

print("\nMissing values:")
print(
    final_external.isna()
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

print("\nSPF present:",
      final_external["spf_present"].sum())

print("DMARC present:",
      final_external["dmarc_present"].sum())

print("DKIM observed:",
      final_external["dkim_observation_found"].sum())

Rows: 10000
Columns: 46
Unique domains: 10000
Duplicate domains: 0

Missing values:
dmarc_adkim               8879
dmarc_aspf                8676
dmarc_subdomain_policy    8404
a_error                   8345
dmarc_percentage          7756
dmarc_fo                  7571
mx_error                  7283
aaaa_records              6975
dmarc_error               6793
spf_all_qualifier         6425
spf_record                6251
spf_error                 4839
dmarc_policy              3322
dmarc_record              3295
aaaa_error                3025
mx_records                2795
a_records                 1655
dmarc_present                0
checked_at_utc               0
domain                       0
dtype: int64

SPF present: 3749
DMARC present: 6705
DKIM observed: 4714


In [ ]:
# ============================================
# DOMAIN COVERAGE VALIDATION
# ============================================

population_domains = set(
    domain_population["domain"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
)

final_domains = set(
    final_external["domain"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
)

missing_from_final = (
    population_domains - final_domains
)

extra_in_final = (
    final_domains - population_domains
)

print(
    "Domains missing from final dataset:",
    len(missing_from_final)
)

print(
    "Unexpected extra domains:",
    len(extra_in_final)
)

Domains missing from final dataset: 0
Unexpected extra domains: 0


In [ ]:
# ============================================
# FINAL EXTERNAL DATASET
# ============================================

final_external_path = os.path.join(
    EXTERNAL_DIR,
    "final_external_deliverability_dataset_10000.csv"
)

final_external.to_csv(
    final_external_path,
    index=False
)

print("FINAL EXTERNAL DATASET SAVED:")
print(final_external_path)

FINAL EXTERNAL DATASET SAVED:
/content/drive/MyDrive/AI_Email_Deliverability_Intelligence/data/external/final_external_deliverability_dataset_10000.csv
